# Trinidad and Tobago GDP nowcasting pipeline

Jamaica beta16 method on the workshop kit's plumbing (build of 2026-10-22). Lives in `code/` of guerreroda/nwcst_tto next to the data notebooks; its knobs are in `config_nwcst.py` (`config.py` belongs to the data notebooks); reads `../data/` and writes `../output/`. Every knob is in `config_nwcst.py`; cells are tagged `# [tto:<name>]` so the runners find them by name.
Runs: `TTO_RUN_TAG=ident` (test window from 2019-Q4, COVID 2020-Q2 as a test quarter, no live readings) and `TTO_RUN_TAG=prod` (test window from 2023-Q2, live quarter 2026-Q3).


## 1. Setup: config and paths


In [ ]:
# [tto:setup]
# CONFIG LOADER -- edit config_nwcst.py, not this notebook. Gives `cfg` (every knob) and `P` (the paths).
import os, importlib.util
CODE_DIR = os.getcwd()
_spec = importlib.util.spec_from_file_location("config", os.path.join(CODE_DIR, "config_nwcst.py"))
cfg = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(cfg)
P = cfg.derive_paths(CODE_DIR)
print(f"[CONFIG] {cfg.COUNTRY} | run {cfg.RUN_TAG} | date_str {cfg.DATE_STR} | "
      f"train {cfg.TRAIN_START} -> test {cfg.TEST_START_ACTIVE} -> end {cfg.TEST_END_ACTIVE or 'last print'} -> live {cfg.DESIRED_DATE_ACTIVE}")


## 2. Data, dates and estimator switches


In [ ]:
# [tto:data]
# =========================================================
# User settings (edit here as necessary)
# =========================================================

import pandas as pd
from datetime import datetime

# ---------------------------------------------------------
# 0) DATA/METADATA FILE NAMES
# ---------------------------------------------------------
data_file, meta_file = cfg.DATA_CSV, cfg.METADATA_CSV   # from config (code/README_pipeline.md)

# ---------------------------------------------------------
# 1) DEFINE TRAIN / TEST WINDOW
# ---------------------------------------------------------
# These dates define the sample splits used for model training and evaluation.
# - Training sample:   [train_start_date, test_start_date)
# - Test sample:       [test_start_date, test_end_date]
train_start_date = cfg.TRAIN_START          # 2013-03-01
test_start_date  = cfg.TEST_START_ACTIVE    # prod 2023-06-01 / ident 2019-12-01

# Convert to Timestamp for safer comparisons later
train_start_date = pd.to_datetime(train_start_date)
test_start_date  = pd.to_datetime(test_start_date)

# ---------------------------------------------------------
# 2) DEFINE OUT-OF-SAMPLE FORECAST TARGET (LAST DATE / VINTAGE)
# ---------------------------------------------------------
# This is the date for which you want to produce the final nowcast/forecast.
# Use the first day of the month (or the appropriate GDP reference month).
desired_date = pd.to_datetime(cfg.DESIRED_DATE_ACTIVE) if cfg.DESIRED_DATE_ACTIVE else None   # None in the identification run

# ---------------------------------------------------------
# 3) DEFINE OUTPUT FILENAMES (timestamped)
# ---------------------------------------------------------
# Timestamp output files so each run creates a unique set of results.
date_str = cfg.DATE_STR   # the orchestrator locks it through TTO_LAUNCH_DATE_STR

forecast_file     = f"{date_str} 3_nwcst Tab fcst.xlsx"   # model forecasts / nowcasts
performance_file  = f"{date_str} 3_nwcst Tab perf.xlsx"   # model performance metrics
outofsample_file  = f"{date_str} 3_nwcst Tab oos.xlsx"    # full OOS prediction panel

# (Optional) If you want to resume / compare against a previous run,
# set last_performance to the filename of the prior performance workbook.
# Example: last_performance = "perf_20251207.xlsx"
last_performance = None

# ---------------------------------------------------------
# 4) MODEL SETTINGS
# ---------------------------------------------------------
# Target GDP series ID
target_variable = cfg.TARGET

# Prediction horizon (vintages) to evaluate:
# - negative values: backcasts (e.g., -1 = one period back)
# - 0: nowcast (current period)
# - positive values: forecasts (e.g., +1 = one period ahead)
lags = list(cfg.LAGS)

# ---------------------------------------------------------
# 5) TARGET MODELS
# ---------------------------------------------------------
# Define models of interest
# ---- pool filter (2026-10-20): which estimated cells enter the regime-aware nowcast, bands, clusters and live quarter ----
# Trinidad (user decision 2026-10-25, after identification run 3): DT4 (4-quarter leaves) replaces DT (2-quarter leaves)
# on every design. On the 12 ordinary test quarters DT4 beat DT at every vintage (RMSE 3.1-3.5 vs 3.7-4.5) and neither
# read COVID; DT is no longer estimated (config RUN["dt"] = 0). Jamaica beta16 keeps its DT/DT4 split by design instead.
# The mask still drops any DT rows found in an older output so that a re-read pool is consistent.
def pool_mask(df):
    est = df["estimator"].astype(str).str.upper()
    return ~(est == "DT")

# vintage column names of the fcst table (same rule as nwcst_helpers.vintage_col_names: the -2..+2 grid keeps the legacy names)
_legacy = {-2: "two_back", -1: "one_back", 0: "zero_back", 1: "one_ahead", 2: "two_ahead"}
VINTAGE_NAMES = {l: _legacy[l] for l in lags} if sorted(lags) == [-2, -1, 0, 1, 2] else {l: f"p{l:+d}" for l in sorted(lags)}
VINTAGES = {name: l for l, name in VINTAGE_NAMES.items()}        # name -> vintage, used by the post and live cells
VINTAGE_COLS = list(VINTAGE_NAMES.values())
def reading_guard(df, cols=VINTAGE_COLS, label=""):
    """Trinidad 2026-10-25 (user decision): a reading beyond +/- READING_GUARD pp (config section 7) is dropped from the
    pool for that quarter and vintage (set to NaN; the long tables drop it). 91 of the 956 production cells, all linear on
    raw designs, produced readings of up to 859 that the weighted mean followed (33.5 at 2024-Q3 V-2, 17.7 at 2024-Q2 V+1).
    No Trinidad quarter has exceeded 12.3 in absolute value, so a true crash passes."""
    df = df.copy(); cols = [c for c in cols if c in df.columns]
    bad = df[cols].abs() > cfg.READING_GUARD; n = int(bad.values.sum())
    df[cols] = df[cols].mask(bad)
    if n: print(f"[reading guard {label}] {n} readings beyond |{cfg.READING_GUARD:g}| dropped ({int(bad.any(axis=1).sum())} cell-quarters)")
    return df
HEADLINE_EXCLUDE = set()     # kept for older cells; the rule now lives in pool_mask
run = dict(cfg.RUN)   # Jamaica knobs, see config section 4

# Parameter: iterations in GBT, DT, RF
_iter_ = cfg.ITER

# =========================================================
# =========================================================
# =========================================================
# Nowcasting pipeline: imports, paths, data loading, and preprocessing
# =========================================================
# This section:
# 1) Imports all required libraries for the nowcasting toolkit (ML + time-series + LSTM)
# 2) Defines local folder paths for raw inputs, processed data, and outputs
# 3) Loads the full dataset and metadata
# 4) Deflates nominal variables using CPI (base year from config)
# 5) Seasonally adjusts series using trend extraction from seasonal decomposition
# 6) Converts all series to growth rates (percent change)
# 7) Cleans the final modeling dataset and prints library versions for reproducibility
# =========================================================

import os, json, time, warnings, re
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# -----------------------------
# Models / ML utilities
# -----------------------------
from sklearn import linear_model, tree
from sklearn.linear_model import (
    LinearRegression, RidgeCV, ElasticNet, Lasso, Lars
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# Deep learning (LSTM nowcasting)
import torch
from nowcast_lstm.LSTM import LSTM

# Statsmodels seasonal adjustment
from statsmodels.tsa.seasonal import seasonal_decompose

# Reduce noisy warnings from some estimators / decomposition
warnings.filterwarnings("ignore", category=UserWarning)

# =========================================================
# Paths
# =========================================================
# Define local project directories (relative to current working directory)
PATH_RAW, PATH_DATA = P["raw"], P["data"]                       # from cfg.derive_paths
PATH_OUTPUT = os.path.join(P["output"], cfg.OUTPUT_SUBFOLDER)    # output/<date>_tto_prod or _ident
# Create folder if it does not exist
os.makedirs(PATH_OUTPUT, exist_ok=True)

# =========================================================
# Load data and metadata
# =========================================================
# Main modeling dataset (indexed by date)
data = pd.read_csv(
    os.path.join(PATH_DATA, data_file ),
    parse_dates=["date"],
    index_col="date"
)

# ---- pseudo-prints (variant switch, 2026-10-25): a nowcast written into the target as if it were the print ----
# Excluded from the test window (never scored, never on the error record); seen by the seasonal adjustment, the
# training sample of the live reading and the lag features (as a published quarter). config.PSEUDO_PRINTS from the
# environment variable TTO_PSEUDO_PRINTS.
PSEUDO_DATES = [pd.Timestamp(k) for k in cfg.PSEUDO_PRINTS]
_real_prints = data[target_variable].dropna().drop(index=[d for d in PSEUDO_DATES if d in data.index], errors="ignore")
test_end_date = _real_prints.index.max()
if cfg.TEST_END_ACTIVE:                                            # identification run stops at 2022-Q4
    test_end_date = min(test_end_date, pd.to_datetime(cfg.TEST_END_ACTIVE))
# The pseudo value is a seasonally adjusted growth rate (%, q/q), the unit of the nowcast; the target column holds the
# NOT seasonally adjusted LEVEL. Conversion with the seasonal component of the classical additive decomposition of the
# level series as it stands (period 4, the same decomposition the filters cell applies):
#     NSA_t = (NSA_{t-1} - s_{t-1}) * (1 + g/100) + s_t      (2026-10-26 fix: the first pass wrote the growth rate into the level)
if cfg.PSEUDO_PRINTS:
    from statsmodels.tsa.seasonal import seasonal_decompose as _sd
    _lvl = _real_prints.sort_index(); _seas = _sd(_lvl, model="additive", extrapolate_trend="freq", period=4).seasonal
    _s_of = lambda d: float(_seas[_seas.index.month == pd.Timestamp(d).month].iloc[-1])
for _d, _v in cfg.PSEUDO_PRINTS.items():
    _d = pd.Timestamp(_d); _prev = _lvl.index[_lvl.index < _d].max()
    if _d in data.index and pd.isna(data.loc[_d, target_variable]):
        _nsa = (float(_lvl.loc[_prev]) - _s_of(_prev)) * (1 + float(_v) / 100) + _s_of(_d)
        data.loc[_d, target_variable] = _nsa
        print(f"[pseudo-print] {target_variable} {_d:%Y-%m}: SA growth {_v:+.3f} % -> NSA level {_nsa:,.1f} (previous print {_prev:%Y-%m} = {float(_lvl.loc[_prev]):,.1f}; not a test quarter)")
    else:
        print(f"[pseudo-print] {_d:%Y-%m} ignored: a real print exists or the date is outside the panel")

# Metadata describing each variable (frequency, nominal/real flags, etc.)
# NOTE: this uses a Windows-style path separator; works on Windows.
# For portability, consider: os.path.join(PATH_DATA, "meta_full.csv")
metadata = pd.read_csv(os.path.join(PATH_DATA, meta_file))

# Key column names and CPI series ID used for deflation
meta_col = "ticket"     # variable identifier column in metadata
cpi = cfg.CPI_TICKER    # CPI series used to deflate nominal variables

# =========================================================
# Deflate nominal variables using CPI (base year from config: 2018 for Trinidad, as the kit and the beta14 run)
# =========================================================
# Many series are nominal; deflating them puts everything in real terms.
base = data[cpi][data.index.year == cfg.CPI_BASE_YEAR].mean()

# Identify variables flagged as nominal in metadata
nominal = metadata.loc[metadata["nominal"] == 1][meta_col].values

# Deflate: Real_t = Nominal_t / CPI_t * CPI_base   (corrected: was multiplying)
for column in data.columns:
    if column in nominal:
        data[column] = data[column] / data[cpi] * base    # beta11: divide (deflate) not multiply


## 3. Filters, redundancy rules on the levels, seasonal adjustment, growth rates


In [ ]:
# [tto:filters]
# =========================================================
# Frequency groups and small-series filter
# =========================================================
quarterly = metadata.loc[metadata["freq"].str.lower().str.contains("q", na=False)][meta_col].values
monthly   = metadata.loc[metadata["freq"].str.lower().str.contains("m", na=False)][meta_col].values
weekly    = metadata.loc[metadata["freq"].str.lower().str.contains("w", na=False)][meta_col].values
daily     = metadata.loc[metadata["freq"].str.lower().str.contains("d", na=False)][meta_col].values

for column in list(data.columns):
    if len(data[column].dropna(how="all", axis=0)) < cfg.MIN_OBS:          # 24 observations (was "<= 23")
        data.drop(column, axis=1, inplace=True)

# =========================================================
# Redundancy rules computed on the LEVELS (method rules added 2026-10-22; Jamaica's panel had no instances)
#   A0 degenerate: constant over the last DEGENERATE_WINDOW observations or fewer than DEGENERATE_MIN_DISTINCT distinct values
#   A  exact duplicates across files: identical values on the overlap, the first ticket is kept
# The typed lists (B mechanical, C identities, D net balances, E by decision) are applied at the pre-selection cell.
# =========================================================
_degenerate = [c for c in data.columns if c != target_variable and (
    data[c].dropna().tail(cfg.DEGENERATE_WINDOW).nunique() <= 1 or data[c].dropna().nunique() < cfg.DEGENERATE_MIN_DISTINCT)]
data = data.drop(columns=_degenerate)
_duplicates = []
_live = [c for c in data.columns if c != target_variable]
for _i, _a in enumerate(_live):
    for _b in _live[_i + 1:]:
        if _b in _duplicates: continue
        _both = data[[_a, _b]].dropna()
        if len(_both) >= cfg.MIN_OBS and np.allclose(_both[_a], _both[_b], rtol=1e-6, atol=1e-9):
            _duplicates.append(_b)
data = data.drop(columns=_duplicates)
print(f"[redundancy A0/A] dropped {len(_degenerate)} degenerate and {len(_duplicates)} duplicate series on the levels")

# =========================================================
# Seasonal adjustment (only for non-SA variables)
# =========================================================
# Variables with sa==1 in metadata are already seasonally adjusted by the source.
# For all others, subtract only the seasonal component (keeps trend + residual/shocks).
non_sa = metadata.loc[metadata["sa"] == 0, meta_col].values

if cfg.SA_METHOD == "x13":
    # Variant switch (2026-10-25): X-13ARIMA-SEATS on every non-SA series, target included -- the Barbados beta15-alpha
    # block, verbatim (US Census binary shipped in the nowcst env). Falls back to the classical decomposition per
    # series when X-13 fails (short, monotonic or degenerate series). Both methods use the full sample for the factors.
    import os as _os, sys as _sys, shutil as _shutil
    from statsmodels.tsa.x13 import x13_arima_analysis
    # The US Census binary is looked up in the running environment (conda-forge package `x13as`), then on the PATH.
    # Without it every series takes the classical fallback below, so the run still completes.
    _x13_candidates = ([_os.path.join(_sys.prefix, "bin", n) for n in ("x13as_ascii", "x13as")]
                       + [_os.path.join(_sys.prefix, "Library", "bin", n) for n in ("x13as_ascii.exe", "x13as.exe")]
                       + [_shutil.which("x13as_ascii"), _shutil.which("x13as")])
    X13_BINARY = next((p for p in _x13_candidates if p and _os.path.exists(p)), None)
    if X13_BINARY is None:
        print("[SA x13] X-13ARIMA-SEATS binary not found (conda install -c conda-forge x13as); "
              "every series takes the classical decomposition in this run.")
    else:
        _os.environ.setdefault("X13PATH", _os.path.dirname(X13_BINARY))
    _n_x13 = _n_fallback = 0
    for column in list(data.columns):
        if column not in non_sa:
            continue
        series = data[column].dropna()
        if len(series) < 36:                       # X-13 needs >= 36 observations to estimate seasonality
            continue
        period = 4 if column in quarterly else 12
        try:
            freq_str = "Q" if period == 4 else "M"
            s_idx = series.copy(); s_idx.index = pd.PeriodIndex(s_idx.index, freq=freq_str)
            if X13_BINARY is None:
                raise RuntimeError("no X-13 binary")
            result = x13_arima_analysis(s_idx, x12path=X13_BINARY, prefer_x13=True, outlier=True, print_stdout=False)
            sa = result.seasadj.copy()
            if len(sa) == len(series):
                sa.index = series.index
            else:
                sa_idx = sa.index.to_timestamp()
                if period == 4: sa_idx = sa_idx + pd.DateOffset(months=2)
                sa.index = sa_idx
            data[column] = sa.reindex(data[column].index).combine_first(data[column]); _n_x13 += 1
        except Exception:
            try:
                decomp = seasonal_decompose(series, model="additive", extrapolate_trend="freq", period=period)
                data[column] = data[column] - decomp.seasonal; _n_fallback += 1
            except Exception:
                pass
    print(f"[SA x13] {_n_x13} series adjusted with X-13ARIMA-SEATS, {_n_fallback} with the classical fallback")
else:
    for column in list(data.columns):
        if column not in non_sa:
            continue
        series = data[column].dropna()
        if len(series) < 24:
            continue
        period = 4 if column in quarterly else 12
        try:
            decomp = seasonal_decompose(
                series,
                model="additive",
                extrapolate_trend="freq",
                period=period,
            )
            data[column] = data[column] - decomp.seasonal
        except Exception:
            pass  # leave as-is if decomposition fails

n_adjusted = sum(1 for c in data.columns if c in non_sa)
print(f"Seasonally adjusted {n_adjusted} non-SA series ({cfg.SA_METHOD}).")

# =========================================================
# Convert levels to growth rates (percent change)
# =========================================================
# Quarterly series: quarter-over-quarter growth rate
# All other series: default percent change at their observed frequency
for column in data.columns:
    if column in quarterly:
        # Get non-NaN values with their original index
        non_nan = data[column].dropna()
        
        # Calculate percent change on non-NaN values only
        pct_change_values = non_nan.pct_change(periods=1, fill_method=None) * 100
        
        # Create a new series filled with NaN, then update with calculated values
        result = pd.Series(np.nan, index=data.index)
        result.loc[pct_change_values.index] = pct_change_values
        
        data[column] = result
    else:
        # Store original NaN mask
        original_was_nan = data[column].isna()
        
        # For monthly/weekly/daily, calculate normally
        pct_change_values = data[column].pct_change(periods=1, fill_method=None) * 100
        
        # Set to NaN where previous value was NaN OR current original value was NaN
        prev_was_nan = data[column].shift(1).isna()
        pct_change_values[prev_was_nan | original_was_nan] = np.nan
        
        data[column] = pct_change_values

# =========================================================
# Drop columns in quarterly with more than 75% missing values
# =========================================================
for column in [ col for col in data.columns if col not in quarterly ] :
    missing_pct = data[data.index<=test_start_date][column].isna().sum() / len(data[data.index<=test_start_date][column])
    if missing_pct > cfg.MAX_MISSING_TRAIN:   # 0.75
        data.drop(column, axis=1, inplace=True)

# =========================================================
# Final cleaning
# =========================================================
# Replace infinities created by pct_change with NaN
data.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop CPI from predictor set (used only for deflation)
#data.drop(cpi, axis=1, inplace=True)

# Keep data from the training start onward and remove fully-missing rows
data = data[~(data.index < train_start_date)].reset_index(drop=False)
data = data.set_index("date").dropna(how="all", axis=0)

# beta15: the October 2026 metadata calls the publication lag `publication_lag`
if "months_lag" not in metadata.columns and "publication_lag" in metadata.columns:
    metadata["months_lag"] = metadata["publication_lag"]

metadata["months_lag"] = (
    pd.to_numeric(metadata["months_lag"], errors="coerce")
      .fillna(1)
      .astype(int)
)

# Drop those columns
data = data.drop(columns=[col for col in data.columns if 'RGDP' in col and col != 'RGDP0000'])
for col in data.columns :
    last = data[col].dropna().index.max().year
    if last < pd.to_datetime(cfg.LAST_OBS_MIN).year :        # 2024
        data.drop(col, axis=1, inplace=True)
# Drop high-publication lag series
tickets_to_keep = metadata[metadata.months_lag <= cfg.MAX_PUB_LAG]['ticket'].tolist()   # 8 months for Trinidad (Jamaica 5)
# Keep only these columns (that exist in data)
data = data[[ col for col in data.columns if col in tickets_to_keep or "RGDP" in col ]]

# =========================================================
# Data-quality guard as a PANEL rule (user decision 2026-10-24, after the identification run of that day):
# a series whose standardised growth rate, against its training-window distribution, exceeds MAX_ABS_Z anywhere
# leaves the candidate panel before the race. Jamaica's guard (pre-selection cell) only kept such series out of the
# shock features and the breadth count; on Trinidad 53 of them (rates near zero, past-due loans, debt flows) sat in
# the panel, 12 inside the selected groups, and blew up the linear cells (ridge readings of +250 at 2022-Q3 V+1).
# =========================================================
_tr_mask = data.index < test_start_date
_pred_cols = [c for c in data.columns if c != target_variable]
_z_panel = (data[_pred_cols] - data.loc[_tr_mask, _pred_cols].mean()) / data.loc[_tr_mask, _pred_cols].std().replace(0, np.nan)
_worst_panel = _z_panel.abs().max()
_extreme = sorted(_worst_panel[_worst_panel > cfg.MAX_ABS_Z].index)
pd.DataFrame({"ticket": _extreme, "max_abs_z": _worst_panel[_extreme].round(1).values,
              "at": _z_panel[_extreme].abs().idxmax().dt.strftime("%Y-%m").values,
              "description": [metadata.set_index(meta_col)["description"].get(t, "") for t in _extreme]}) \
  .sort_values("max_abs_z", ascending=False).to_csv(os.path.join(PATH_OUTPUT, f"{date_str} 3_nwcst Tab panel_guard.csv"), index=False)
data = data.drop(columns=_extreme)
print(f"[panel guard] dropped {len(_extreme)} series with |z| > {cfg.MAX_ABS_Z} against the training window "
      f"(families: {pd.Series([t[:4] for t in _extreme]).value_counts().head(6).to_dict()}); panel now {data.shape[1] - 1} candidates")

data.to_csv(os.path.join(PATH_OUTPUT, "data_processed.csv"), index=True)
metadata = metadata[metadata.ticket.isin(data.columns)]
metadata.to_csv(os.path.join(PATH_OUTPUT, "meta_processed.csv"), index=True)
# =========================================================
# Reproducibility: print core library versions
# =========================================================
print("================== Library Versions =================")
import sklearn, numpy, pandas
print("scikit-learn:", sklearn.__version__)
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)


# Data dimensions (observations and variables)
print("Data dimensions:", data.shape)
print(f"Training dates: {train_start_date.strftime('%Y-%m-%d')} to {(test_start_date - pd.DateOffset(days=1)).strftime('%Y-%m-%d')}")
T1 = data[data.index < test_start_date].shape[0]
T2 = data[(data.index >= test_start_date) & (data.index <= test_end_date)].shape[0]
print(f"Training periods: {T1} ({T1/(T1+T2) :.2%})")
print(f"Testing dates: {test_start_date.strftime('%Y-%m-%d')} to {test_end_date.strftime('%Y-%m-%d')}")
print(f"Testing periods: {T2} ({T2/(T1+T2) :.2%})")


## 4. Model library


In [ ]:
# [tto:helpers]
# The model library lives in nwcst_helpers.py, beside this notebook (kit module + three Jamaica beta16 estimators).
# joblib/loky workers re-import the module, so its folder must be importable -- hence the sys.path insert.
import sys, os
for _p in (os.getcwd(), CODE_DIR):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import nwcst_helpers
from nwcst_helpers import *   # estimate_factors, lars_variable_ranking, rank_*, gen_lagged_data, flatten_data,
                              # ffill_dataset, lagged_target, perform_tab, forecast_table, fit_/predict_/oos_ ...


## 5. Pre-selection: redundancy lists, six-screen race, groups, shock features


In [ ]:
# [tto:preselection]
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------
# BETA15 PRESELECTION HORSE RACE
# LARS (incumbent) vs SIS, T-STAT, SHAP, OCMT, iBMA -- rankers from BoJ's pilot notebook.
# Steps: 1) Aggregate to quarterly (beta14b, unchanged)
#        2) Rank ONCE per method on the panel + ROLLING per test date (beta12 logic, now per method)
#        3) Time-series CV evaluation of every (method, size) vs the all-variables PCA+OLS benchmark
#           (beta14b evaluator, unchanged: same folds, same benchmark for every method)
#        4) Ratio curves, overlap vs LARS, overlap matrix at k=60, head-to-head table, xlsx export
#        5) spec_list per method: ratio < 1, soft-fallback top-15 (beta14b), optional cap
#        6) Shock features (N/R) + SHOK0000 (beta5/beta6, unchanged)
# ---------------------------------------------
factors = cfg.FACTORS   # 1: Bai-Ng IC1 consistently selects 1 factor given our sample size

# 1. Aggregate to quarterly (no qmin/qrange here — keeps benchmark fair)
try:
    agg_map = metadata.set_index('ticket')['agg_method'].to_dict()
    agg_funcs = {
        col: (agg_map[col] if agg_map[col] in ['mean', 'last', 'sum'] else 'mean')
        for col in data.columns if col in agg_map
    }
    pre_selected = data.resample('QS').agg(agg_funcs)
except:
    pre_selected = data.resample('QS').mean()

pre_selected.index = pd.to_datetime(pre_selected.index)
pre_selected = pre_selected.dropna(how='all', axis=1)
pre_selected = pre_selected.loc[:, pre_selected.isnull().mean() <= 0.75]
pre_selected = pre_selected.ffill().bfill()
pre_selected = pre_selected[pre_selected.index <= data[target_variable].dropna().drop(index=[d for d in PSEUDO_DATES if d in data.index], errors="ignore").index.max()]   # last REAL print (2026-10-26): a pseudo quarter never enters the race

print(f"Pre-selected shape (quarterly): {pre_selected.shape}")

# =========================================================
# beta15 redundancy rules -- applied to the quarterly panel BEFORE any ranking
# =========================================================
CORR_PRUNE  = cfg.CORR_PRUNE    # 0.8, 1b: max |corr| allowed with a variable already kept in a ranking
SHOCK_SHARE = cfg.SHOCK_SHARE   # 0.05: SHOK0000 fires when >= this share of the monthly qmin variables (floor 4) break -3 sigma
SMALL_SHARE = cfg.SMALL_SHARE   # 0.20, 1c: a component below this share of its aggregate is dropped

# 1a. mechanical derivations of other series (typed list B in config: 30-day NTL windows, yoy credit growth, debt ratios, import cover)
_mechanical = [c for c in pre_selected.columns if c in cfg.MECHANICAL]

# 1c. additive identities (typed list C in config, verified parts/aggregate = 1 on the October 2026 levels)
IDENTITIES = cfg.IDENTITIES
_levels = pd.read_csv(os.path.join(PATH_DATA, data_file), parse_dates=["date"], index_col="date")
_small = sorted({p for a, parts in IDENTITIES.items() if a in _levels
                   for p in parts if p in _levels and (_levels[p] / _levels[a]).mean() < SMALL_SHARE})
_nets = [c for c in pre_selected.columns if c in cfg.NET_BALANCES]                 # 1d (new rule): signed net balances, differences of gross flows in the panel
_excluded_by_decision = list(cfg.EXCLUDED_BY_DECISION)                          # 1e: typed list E in config, reasons logged
_drop = [c for c in dict.fromkeys(_mechanical + _small + _nets + _excluded_by_decision) if c in pre_selected.columns]
pre_selected = pre_selected.drop(columns=_drop)
print(f"[redundancy] dropped {len(_drop)} series: {sum(c in _mechanical for c in _drop)} mechanical twins, "
      f"{sum(c in _small for c in _drop)} small identity components {sorted(c for c in _drop if c in _small)}, "
      f"{sum(c in _nets for c in _drop)} net balances, "
      f"{sum(c in _excluded_by_decision for c in _drop)} by decision {_excluded_by_decision}")
print(f"[redundancy] panel now {pre_selected.shape[1] - 1} candidates")

def _prune(ranking, panel, thr=CORR_PRUNE):
    """1b. Walk down a ranking; keep a variable only if |corr| with every variable already kept is below thr."""
    cols = [v for v in ranking if v in panel.columns]
    C = panel[cols].corr().abs()
    kept = []
    for v in cols:
        if not kept or C.loc[v, kept].max() < thr:
            kept.append(v)
    return kept

# ---------------------------------------------
# 2. Rankings
# ---------------------------------------------
import numpy as np
METHODS          = ["lars", "sis", "tstat", "shap", "ocmt", "ibma"]
FIXED_SIZE       = min(cfg.FIXED_SIZE_MAX, pre_selected.shape[1] - 1)
MAX_ROLLING_VARS = cfg.MAX_ROLLING_VARS
gdp_lags_screen  = cfg.GDP_LAGS_SCREEN
hac_lags         = cfg.HAC_LAGS
RANK_ON_TRAIN_ONLY = cfg.RANK_ON_TRAIN_ONLY  # False = beta14b convention (static ranking on the full quarterly panel)
                            # True  = pilot convention (training window only)

def _rank_all(panel, n_max):
    """One ordered list per method for the given quarterly panel (best first)."""
    out = {}
    out["lars"], _    = lars_variable_ranking(panel, target=target_variable,
                                               max_vars=min(n_max, panel.shape[1] - 1),
                                               test=None, factors=factors)
    out["sis"], _     = rank_sis(panel, target_variable)
    out["tstat"], _   = rank_tstat(panel, target_variable, gdp_lags=gdp_lags_screen, hac_lags=hac_lags)
    out["shap"], _    = rank_shap(panel, target_variable)
    out["ocmt"], _, _ = rank_ocmt(panel, target_variable, gdp_lags=gdp_lags_screen, hac_lags=hac_lags)
    out["ibma"], _    = rank_ibma(panel, target_variable)
    return {m: _prune(list(v), panel)[:n_max] for m, v in out.items()}   # 1b: prune static AND rolling rankings

# 2a. Rolling: at each test quarter, rank with data strictly BEFORE that quarter (beta12 logic, per method)
_test_dates = [d for d in pre_selected.index
               if d >= pd.Timestamp(test_start_date) and d <= pre_selected.index.max()]
print(f"[Rolling] test dates: {len(_test_dates)} ({_test_dates[0].date()} -> {_test_dates[-1].date()})")

rolling_selections = {m: {} for m in METHODS}       # {method: {Timestamp: [ordered vars]}}
_t_roll = time.time()
for _t in _test_dates:
    _train_slice = pre_selected[pre_selected.index < _t]
    if len(_train_slice) < 16:
        for m in METHODS:
            rolling_selections[m][_t] = []
        continue
    _r = _rank_all(_train_slice, MAX_ROLLING_VARS)
    for m in METHODS:
        rolling_selections[m][_t] = _r[m]
    print(f"  [Rolling] {_t.date()}: " + ", ".join(f"{m}={len(_r[m])}" for m in METHODS), flush=True)
print(f"[Rolling] done in {time.time() - _t_roll:.0f}s")

# 2b. Static ranking (drives the size selection through the CV below)
_rank_panel = pre_selected[pre_selected.index < test_start_date] if RANK_ON_TRAIN_ONLY else pre_selected
rankings = _rank_all(_rank_panel, FIXED_SIZE)
for m in METHODS:
    print(f"{m.upper():6s} ranked {len(rankings[m]):3d} variables; top-5: {rankings[m][:5]}")

# ---------------------------------------------
# 3. Time-series CV evaluation (expanding window, no leakage) -- beta14b evaluator, unchanged
# ---------------------------------------------
n         = len(pre_selected)
min_train, step, eval_size = cfg.CV_MIN_TRAIN, cfg.CV_STEP, cfg.CV_EVAL_SIZE   # 16, 2, 4

fold_starts = list(range(min_train, n - eval_size, step))
print(f"Number of CV folds: {len(fold_starts)}")

def _cv_mse(cols):
    """Average fold MSE of PCA(factors)+OLS on `cols` (None = all variables, i.e. the benchmark)."""
    mses = []
    for fs in fold_starts:
        fold_train = pre_selected.iloc[:fs]
        fold_val   = pre_selected.iloc[fs:fs + eval_size]
        y_tr = fold_train[target_variable].dropna()
        y_vl = fold_val[target_variable].dropna()
        if len(y_tr) < 5 or len(y_vl) < 1:
            continue
        if cols is None:
            X_tr = fold_train.drop(target_variable, axis=1).loc[y_tr.index].ffill().bfill()
            X_vl = fold_val.drop(target_variable, axis=1).loc[y_vl.index].ffill().bfill()
        else:
            X_tr = fold_train[cols].loc[y_tr.index].ffill().bfill()
            X_vl = fold_val[cols].loc[y_vl.index].ffill().bfill()
        sc    = StandardScaler()
        ncomp = min(factors, X_tr.shape[1], len(y_tr) - 1)
        pca   = PCA(n_components=ncomp).fit(sc.fit_transform(X_tr))
        X_tr_p = pca.transform(sc.fit_transform(X_tr))
        X_vl_p = pca.transform(sc.transform(X_vl))
        m = LinearRegression().fit(X_tr_p, y_tr)
        mses.append(np.mean((y_vl.values - m.predict(X_vl_p)) ** 2))
    return np.mean(mses)

benchmark = _cv_mse(None)
print(f"Benchmark CV MSE: {benchmark:.4f}")

results = []
for m in METHODS:
    for size in range(5, min(FIXED_SIZE, len(rankings[m])) + 1):
        top_k     = rankings[m][:size]
        available = [v for v in top_k if v in pre_selected.columns]
        if len(available) < 2:
            continue
        avg_mse = _cv_mse(available)
        results.append({'method': m, 'size': size, 'selected_vars': ' '.join(top_k),
                        'rmse': avg_mse, 'ratio': avg_mse / benchmark})
d = pd.DataFrame(results).reset_index(drop=True)
print("Sizes beating the benchmark, by method:")
print(d[d['ratio'] < 1].groupby('method')['size'].count().reindex(METHODS).fillna(0).astype(int).to_string())

# ---------------------------------------------
# 4a. Ratio curves, one line per method (replaces beta14b's single LARS curve)
# ---------------------------------------------
ratio_curves = d.pivot(index='size', columns='method', values='ratio').reindex(columns=METHODS)
styles = {"lars": ("black", "-"), "sis": ("gray", "--"), "tstat": ("black", "-."),
          "shap": ("gray", ":"), "ocmt": ("steelblue", "-"), "ibma": ("firebrick", "--")}
fig, ax = plt.subplots(figsize=(8, 4))
for m in METHODS:
    color, ls = styles[m]
    ratio_curves.loc[ratio_curves.index > 10, m].plot(ax=ax, color=color, linestyle=ls, label=m.upper())
ax.axhline(y=1, color='black', linestyle=':', linewidth=1)
ax.set_ylabel("Ratio of CV MSE (selection / all-variables benchmark)")
ax.set_xlabel("Number of variables")
ax.legend(frameon=False, ncol=3)
plt.tight_layout()
plt.savefig(f"{PATH_OUTPUT}/{date_str} 3_nwcst Fig preselection_ratio.png", dpi=300, bbox_inches='tight')
plt.show()

# ---------------------------------------------
# 4b. Overlap with the incumbent (LARS) at every size + full matrix at k=60
# ---------------------------------------------
overlap = []
for size in range(5, FIXED_SIZE + 1):
    row = {"size": size}
    for m in METHODS[1:]:
        sa, sb = set(rankings["lars"][:size]), set(rankings[m][:size])
        row[f"lars-{m}"] = len(sa & sb) / size
    overlap.append(row)
overlap = pd.DataFrame(overlap).set_index("size")
fig, ax = plt.subplots(figsize=(8, 4))
for col, m in zip(overlap.columns, METHODS[1:]):
    color, ls = styles[m]
    overlap[col].plot(ax=ax, color=color, linestyle=ls, label=col.upper())
ax.set_ylabel("Share of common variables vs LARS"); ax.set_xlabel("Number of variables"); ax.set_ylim(0, 1)
ax.legend(frameon=False, ncol=3)
plt.tight_layout()
plt.savefig(f"{PATH_OUTPUT}/{date_str} 3_nwcst Fig preselection_overlap.png", dpi=300, bbox_inches='tight')
plt.show()

k_ref = min(cfg.K_REF, FIXED_SIZE)
ov_mat = pd.DataFrame(index=METHODS, columns=METHODS, dtype=float)
for a in METHODS:
    for b in METHODS:
        ov_mat.loc[a, b] = len(set(rankings[a][:k_ref]) & set(rankings[b][:k_ref])) / k_ref
print(f"Overlap matrix at k={k_ref} (share of common variables):")
print(ov_mat.round(2))

# ---------------------------------------------
# 4c. Head-to-head summary
# ---------------------------------------------
_rows = []
for m in METHODS:
    g = d[d['method'] == m]
    if g.empty:
        continue
    _rows.append({"method": m,
                  "specs_below_threshold": int((g['ratio'] < 1).sum()),
                  "best_ratio": g['ratio'].min(),
                  "best_size": int(g.loc[g['ratio'].idxmin(), 'size']),
                  "median_ratio": g['ratio'].median()})
summary = pd.DataFrame(_rows).set_index("method")
print("Head-to-head summary (lower ratio = better):")
print(summary)

# ---------------------------------------------
# 4d. Export (ratio curves, MSE levels, overlap, summary, run context)
# ---------------------------------------------
with pd.ExcelWriter(f"{PATH_OUTPUT}/{date_str} 3_nwcst preselection_ratio.xlsx", engine="openpyxl") as writer:
    ratio_curves.round(6).to_excel(writer, sheet_name="ratio_curves")
    d.pivot(index="size", columns="method", values="rmse").reindex(columns=METHODS).round(6).to_excel(writer, sheet_name="mse_levels")
    overlap.round(4).to_excel(writer, sheet_name="overlap_vs_lars")
    ov_mat.round(4).to_excel(writer, sheet_name=f"overlap_k{k_ref}")
    summary.to_excel(writer, sheet_name="summary")
    pd.DataFrame({"item": ["benchmark_cv_mse", "factors_pca", "train_start", "test_start", "test_end",
                           "n_variables_panel", "rank_on_train_only", "n_test_dates_rolling"],
                  "value": [benchmark, factors, str(train_start_date), str(test_start_date), str(test_end_date),
                            pre_selected.shape[1] - 1, RANK_ON_TRAIN_ONLY, len(_test_dates)]}
                 ).to_excel(writer, sheet_name="run_info", index=False)
print(f"Preselection tables exported to {PATH_OUTPUT}/{date_str} 3_nwcst preselection_ratio.xlsx")

# ---------------------------------------------
# 5. spec_list: per method, sizes that beat the benchmark (ratio < 1); soft-fallback to the 15 best
#    sizes by CV MSE when fewer than 15 beat it (beta14b); optional cap per method.
# ---------------------------------------------
chosen_methods       = cfg.CHOSEN_METHODS         # None = every method with at least one benchmark-beating group
max_specs_per_method = cfg.MAX_SPECS_PER_METHOD   # a method with MORE beating groups than this is thinned by _spread_pick

def _spread_pick(sl, n):
    """Option 2: best group (lowest CV MSE) in each of n equal-width bins of K; empty bins filled with the best remaining."""
    if len(sl) <= n:
        return sl
    edges = np.linspace(sl['size'].min(), sl['size'].max() + 1e-9, n + 1)
    picks = [b.loc[b['rmse'].idxmin()] for lo, hi in zip(edges[:-1], edges[1:])
             for b in [sl[(sl['size'] >= lo) & (sl['size'] < hi)]] if len(b)]
    p = pd.DataFrame(picks)
    if len(p) < n:
        p = pd.concat([p, sl[~sl.index.isin(p.index)].nsmallest(n - len(p), 'rmse')])
    return p.sort_values('size')

_parts = []
for m in (chosen_methods or METHODS):
    dm = d[d['method'] == m]
    sl = dm[dm['ratio'] < 1].copy()
    if len(sl) == 0:
        print(f"[skip] {m}: no size beats the benchmark")
        continue
    if max_specs_per_method is not None:
        sl = _spread_pick(sl, max_specs_per_method)   # thins only methods with more groups than the cap
    _parts.append(sl)
if not _parts:   # beta14b guard: nothing beats the benchmark -> the 15 best sizes overall by CV MSE
    print("[soft-fallback] no method beats the benchmark; using the 15 best sizes overall")
    _parts.append(d.nsmallest(15, 'rmse').copy())
spec_list = pd.concat(_parts, axis=0).reset_index(drop=True)
top_vars = spec_list.loc[spec_list['rmse'].idxmin(), 'selected_vars'].split(' ')
print(f"Total specifications selected: {len(spec_list)} "
      f"({spec_list['method'].value_counts().reindex(METHODS).fillna(0).astype(int).to_dict()})")

# ---- beta15: consensus groups across the benchmark-beating methods (static, aggregate-level by construction) ----
_beating = [m for m in METHODS if (d[d['method'] == m]['ratio'] < 1).any()]
_sets = {m: set(d[(d['method'] == m) & (d['ratio'] < 1)].sort_values('size').iloc[-1]['selected_vars'].split()) for m in _beating}
_count = pd.Series([v for s_ in _sets.values() for v in s_]).value_counts()
common_vars = []
KEEP_COMMON = cfg.KEEP_COMMON   # the common group is a fixed row like the economist sets (it is the watchlist reference);
                     # its CV ratio is reported, not applied. The 'shared' group stays under the ratio rule.
for _name, _min in [("common", len(_beating)), ("shared", max(len(_beating) - 1, 2))]:   # all methods / all but one
    _vars = sorted(_count[_count >= _min].index,
                   key=lambda v: sum(rankings[m].index(v) + 1 for m in _beating if v in rankings[m]))
    if len(_vars) >= 3 and (_name == "common" or len(_vars) > len(common_vars)):
        _mse = _cv_mse(_vars)
        _fixed = (_name == "common" and KEEP_COMMON)
        if _mse / benchmark < 1 or _fixed:
            spec_list = pd.concat([spec_list, pd.DataFrame([{'method': _name, 'size': len(_vars), 'selected_vars': ' '.join(_vars),
                                                             'rmse': _mse, 'ratio': _mse / benchmark}])], ignore_index=True)
            _how = "kept as a fixed row (ratio reported, not applied)" if (_fixed and _mse / benchmark >= 1) else "kept"
            print(f"[{_name}] {len(_vars)} variables in >= {_min} of {len(_beating)} beating methods: {_vars}  CV ratio {_mse / benchmark:.2f} -> {_how}")
        else:
            print(f"[{_name}] {len(_vars)} variables in >= {_min} of {len(_beating)} beating methods: CV ratio {_mse / benchmark:.2f} -> NOT kept (does not beat the benchmark)")
    if _name == "common":
        common_vars = _vars

# ---- Beta12: each rolling spec references the per-date selection of ITS method, sliced at K ----
spec_list = spec_list.rename(columns={'size': 'K'})
_roll = spec_list['method'].isin(rolling_selections.keys())
spec_list.loc[_roll, 'selected_vars'] = spec_list[_roll].apply(
    lambda r: ' '.join(rolling_selections[r['method']][_test_dates[-1]][:int(r['K'])]), axis=1)
spec_list['rolling_enabled'] = _roll.astype(int)
print(f"[Beta12] spec_list has {len(spec_list)} (method, K) specs (rolling enabled)")

# ---- frozen groups (config section 3, user decision 2026-10-25): static rows outside the race, five designs like any group ----
if getattr(cfg, "FROZEN_GROUPS", None):
    _frz_rows, _frz_missing = [], {}
    for g in cfg.FROZEN_GROUPS:
        _have = [t for t in g["selected_vars"].split() if t in pre_selected.columns]
        _frz_missing[g["spec"]] = [t for t in g["selected_vars"].split() if t not in pre_selected.columns]
        _frz_rows.append({"method": g["method"], "K": int(g["K"]), "selected_vars": " ".join(_have), "rmse": np.nan, "ratio": np.nan, "rolling_enabled": 0,
                          "augment": int(bool(g.get("augment", False)))})   # 2026-10-26: 0 = race design (base tickets only)
    spec_list = pd.concat([spec_list, pd.DataFrame(_frz_rows)], ignore_index=True)
    print(f"[frozen] {len(_frz_rows)} frozen groups appended (static); tickets missing from this panel: {_frz_missing}")
print(f"Best-CV spec: {spec_list.loc[spec_list['rmse'].idxmin(), 'method'].upper()} with {len(top_vars)} variables")

# ---- End beta7 ----

# ---- Beta5: attach shock features to LARS-selected monthly vars ----
# Naming convention follows existing tickets (alphanumeric, no underscores):
#   <8-char-base>N  -> quarterly MINimum  (e.g. MLOAB004 -> MLOAB004N)
#   <8-char-base>R  -> quarterly Range    (e.g. MLOAB004 -> MLOAB004R)

all_base_vars = set()
for _, row in spec_list.iterrows():
    all_base_vars.update(row['selected_vars'].split())

base_monthly_vars = [v for v in all_base_vars if v in monthly]

# ---- beta15 data-quality guard (2026-10-18) ----
# A monthly series whose standardised growth, against its own training-window distribution, exceeds
# MAX_ABS_Z anywhere is treated as broken (level shift, reclassification, series near zero). It stays in
# the candidate panel (penalised estimators and trees cope with outliers) but is excluded from the
# within-quarter shock features and from the breadth indicator, and listed for the data team.
MAX_ABS_Z = cfg.MAX_ABS_Z   # 20
_tr = data.index < test_start_date
_z_full = (data[base_monthly_vars] - data[base_monthly_vars].loc[_tr].mean()) \
          / data[base_monthly_vars].loc[_tr].std().replace(0, np.nan)
_worst = _z_full.abs().max()
broken_vars = sorted(_worst[_worst > MAX_ABS_Z].index)
_desc = metadata.set_index("ticket")["description"]
pd.DataFrame({"ticket": _worst.index, "max_abs_z": _worst.values.round(1),
              "at": _z_full.abs().idxmax().dt.strftime("%Y-%m").values,
              "description": [_desc.get(t, "") for t in _worst.index],
              "excluded_from_shock_features": _worst.values > MAX_ABS_Z}) \
  .sort_values("max_abs_z", ascending=False) \
  .to_csv(os.path.join(PATH_OUTPUT, f"{date_str} 3_nwcst Tab data_quality.csv"), index=False)
print(f"[data quality] {len(broken_vars)} series beyond |z| > {MAX_ABS_Z} excluded from shock features "
      f"and the breadth count: {broken_vars}")
base_monthly_vars = [v for v in base_monthly_vars if v not in broken_vars]
print(f"Computing shock features for {len(base_monthly_vars)} LARS-selected monthly variables")

if base_monthly_vars:
    md = data[base_monthly_vars]

    qmin_full = md.resample('QS').min().reindex(data.index, method='ffill')
    qmin_full.columns = [c + 'N' for c in qmin_full.columns]

    qrange_full = (md.resample('QS').max() - md.resample('QS').min()).reindex(data.index, method='ffill')
    qrange_full.columns = [c + 'R' for c in qrange_full.columns]

    data = pd.concat([data, qmin_full, qrange_full], axis=1)

    # Synthesize metadata rows so gen_lagged_data() knows publication lags
    aug_cols = list(qmin_full.columns) + list(qrange_full.columns)
    aug_meta_rows = []
    for col in aug_cols:
        parent = col[:-1]   # strip final 'N' or 'R'
        parent_row = metadata[metadata['ticket'] == parent]
        if len(parent_row) > 0:
            new_row = parent_row.iloc[0].copy()
            new_row['ticket'] = col
            suffix_label = 'quarterly min' if col.endswith('N') else 'quarterly range'
            new_row['description'] = str(new_row.get('description', col)) + f' ({suffix_label})'
            aug_meta_rows.append(new_row)
    if aug_meta_rows:
        metadata = pd.concat([metadata, pd.DataFrame(aug_meta_rows)], ignore_index=True)

    # Expand each spec's selected_vars: base + baseN + baseR
    def _augment_spec_vars(vars_str):
        out = []
        for v in vars_str.split():
            out.append(v)
            if v in monthly:
                out.append(v + 'N')
                out.append(v + 'R')
        return ' '.join(out)

    # 2026-10-26: frozen groups with augment=0 keep the base tickets only, the design the rolling race cells use
    # (the loop takes the top-K base tickets per quarter, without N/R companions or SHOK0000); augment=1 = static design.
    _aug = spec_list['augment'].fillna(1).astype(bool) if 'augment' in spec_list.columns else pd.Series(True, index=spec_list.index)
    spec_list.loc[_aug, 'selected_vars'] = spec_list.loc[_aug, 'selected_vars'].apply(_augment_spec_vars)
    top_vars = spec_list.loc[spec_list['rmse'].idxmin(), 'selected_vars'].split(' ')

    print(f"After augmentation: best spec has {len(top_vars)} features")

    # ---- Beta6: is_shock binary feature (Path C); beta15: share-based threshold ----
    # Shock flag: at least SHOCK_SHARE of the qmin variables (floor 4) are below -3 sigma of their
    # training-period distribution at the same time. beta6's absolute '4' did not scale with the
    # inventory: verified 2026-10-17 it flagged 6 quarters on the April data (99 vars) and 13 on the
    # October data (140 vars); 5 % flags COVID 2020-Q2 and Melissa 2025-Q4 only on the October data.
    _train_mask = qmin_full.index < test_start_date
    _mu  = qmin_full.loc[_train_mask].mean()
    _sig = qmin_full.loc[_train_mask].std().replace(0, float("nan"))
    _z   = (qmin_full - _mu) / _sig
    _n_broken = (_z < -3).sum(axis=1).fillna(0).astype(int)
    _shock_thr = max(4, int(np.ceil(SHOCK_SHARE * qmin_full.shape[1])))
    _shock_col = (_n_broken >= _shock_thr).astype(int)
    print(f"  [is_shock] threshold {_shock_thr} of {qmin_full.shape[1]} monthly variables")
    _shock_col.name = 'SHOK0000'
    data['SHOK0000'] = _shock_col.reindex(data.index).fillna(0).astype(int)
    print(f"  [is_shock] flagged {int(_shock_col.sum())} rows across {_shock_col.groupby(_shock_col.index.to_period('Q')).max().sum()} quarters")

    # Synthetic metadata row so gen_lagged_data knows publication lag (treat as daily: 0 months lag)
    _shock_meta = metadata.iloc[0].copy()
    _shock_meta['ticket'] = 'SHOK0000'
    _shock_meta['series'] = 'shock_indicator'
    _shock_meta['description'] = 'beta6 binary shock indicator'
    _shock_meta['freq'] = 'daily'
    _shock_meta['months_lag'] = 0
    _shock_meta['agg_method'] = 'max'
    metadata = pd.concat([metadata, pd.DataFrame([_shock_meta])], ignore_index=True)

    # Add SHOK0000 to every spec's feature list
    spec_list.loc[_aug, 'selected_vars'] = spec_list.loc[_aug, 'selected_vars'].apply(lambda s: s + ' SHOK0000')   # 2026-10-26: not for base-only frozen groups
    top_vars = spec_list.loc[spec_list['rmse'].idxmin(), 'selected_vars'].split(' ')
    print(f"Beta6: added SHOK0000. Total shock-quarters flagged: {int(data['SHOK0000'].sum())}")
    # ---- End beta6 ----

# ---- End beta5 ----

pre_selected_columns_monthly = [c for c in pre_selected.columns if c in monthly]   # beta15: fixed panel for the real-time regime (cell 23)
del pre_selected

# Save estimation parameters
parameters = {
    "train_start": train_start_date.strftime("%Y-%m-%d"),
    "test_start": test_start_date.strftime("%Y-%m-%d"),
    "test_end": test_end_date.strftime("%Y-%m-%d"),
    "lars_factors": factors,
    "corr_prune": CORR_PRUNE,
    "small_share": SMALL_SHARE,
    "shock_share": SHOCK_SHARE,
    "max_abs_z": MAX_ABS_Z,
    "excluded_from_shock_features": broken_vars,
    "excluded_by_decision": _excluded_by_decision,
    "net_balances_dropped": _nets,
    "n_dropped_redundant": len(_drop),
    "penalty_cv": "ElasticNetCV(l1_ratio 0.25/0.5/0.75, 50 alphas) + LassoCV(50 alphas), TimeSeriesSplit(4)",
    "consensus_groups": {"common": common_vars},
    "preselection_methods": METHODS,
    "preselection_chosen": chosen_methods,
    "max_specs_per_method": max_specs_per_method,
    "rank_on_train_only": RANK_ON_TRAIN_ONLY,
    "screen_gdp_lags": gdp_lags_screen,
    "screen_hac_lags": hac_lags,
    "shap_model": "GBR(n_estimators=300, max_depth=2, lr=0.05, seed=0)",
    "ocmt_params": "alpha=0.05, delta=1, delta_star=1.5",
    "ibma_params": "w=10, pip_drop=0.05, BIC weights",
    "model_iterations": int(_iter_),
    **run
}
with open(os.path.join(PATH_OUTPUT, "estimation_parameters.json"), "w") as f:
    json.dump(parameters, f, indent=2)

print("Distribution of CV MSE in the selected specs, by method:")
print(spec_list.groupby('method')['rmse'].describe())
metadata[metadata['ticket'].isin(top_vars)]


## 6. Specifications, estimator map, economist groups


In [ ]:
# [tto:specs]
import warnings
warnings.filterwarnings('ignore')
# --- Base / benchmark naming (beta15): <method><K>, e.g. lars43, shap28, ocmt46;
#     the single best CV spec is "bench<K>" (beta14b convention) ---
spec_list = spec_list.copy().reset_index(drop=True)

spec_list["spec"] = spec_list["method"] + spec_list["K"].astype(int).astype(str)
bench_idx = spec_list["rmse"].idxmin()
spec_list.loc[bench_idx, "spec"] = "bench" + str(int(spec_list.loc[bench_idx, "K"]))

# --- Base settings (data_flat=3 only) ---
base = spec_list.assign(gdp_lags=0, PCA=0, data_flat=3)

variants = [base]

# --- Flats branch (data_flat=4 only, no PCA/lag variants); suffix so the method stays visible ---
flats = base.assign(data_flat=4)
flats["spec"] = flats["spec"] + "flats"
variants.append(flats)

# --- PCA variants ONLY on base (data_flat=3) ---
p5 = base.assign(PCA=5)
p5["spec"] = p5["spec"] + "p5"
variants.append(p5)

p3 = base.assign(PCA=3)
p3["spec"] = p3["spec"] + "p3"
variants.append(p3)

# --- Lags variant ONLY for p5 specs on base (data_flat=3) ---
p5_l2 = p5.copy()
p5_l2["gdp_lags"] = 2
p5_l2["spec"] = p5_l2["spec"] + "l2"
variants.append(p5_l2)

# --- Final spec_list ---
spec_list = pd.concat(variants, axis=0).reset_index(drop=True)
spec_list = spec_list[(spec_list["PCA"] == 0) | (spec_list["PCA"] < spec_list["K"])].reset_index(drop=True)   # beta15: PCA needs fewer factors than variables

# --- beta15: estimator map per variant (evidence: August 2026 run, estimator x variant medians) ---
#   linear models (ridge, elastic net, LASSO, OLS if enabled): factor designs only -- on the raw designs
#       (111-148 regressors on ~33 quarters) their August RMSE was 10-17 vs 2.6-3.2 on p3/p5;
#   trees: every design -- forests and boosting best on base/flats; the single tree on factor designs recalls the
#       pandemic leaf (readings of -16.7) in shock AND some ordinary quarters; kept on purpose (2026-10-18): the
#       regime-aware weights silence it in ordinary quarters (quiet RMSE identical with or without, tested on the
#       October pool) and it is the main carrier of pandemic memory into the shock-regime consensus;
#   linear on stacked-lag designs: restored 2026-10-18 for the next run -- their exclusion rested on the August
#       fixed-penalty evidence (alpha=1e-5, i.e. OLS on p>n designs); with CV penalties it must be re-tested;
#   LSTM: only where the design differs for it -- it ignores flattening (flats == base) and GDP lags (l2 == p5).
# The worker runs only the estimators listed here; an empty/missing value means "all enabled".
LINEAR_EST, TREE_EST, LSTM_EST = ["ols", "olsr", "enet", "lasso"], ["dt", "dt4", "rf", "gbt"], ["lstm"]
# 2026-10-19: decide by the row's design columns, not by the end of its name. The name test misread 'shap5'
# (a base-design group of five variables) as a five-factor design and gave it the linear estimators in the
# October run (3 extra cells: 882 instead of 879); any future group of size 3 or 5 would have hit the same.
def _estimators_for(row):
    if row["PCA"] > 0 and row["gdp_lags"] > 0:  return LINEAR_EST + TREE_EST              # p5l2
    if row["PCA"] > 0:                          return LINEAR_EST + TREE_EST + LSTM_EST   # p3, p5, economist sets
    if row["data_flat"] == 4:                   return LINEAR_EST + TREE_EST              # flats
    return LINEAR_EST + TREE_EST + LSTM_EST        # base: LSTM reads the raw monthly sequences here (not the stacked table)
spec_list["estimators"] = spec_list.apply(lambda r: " ".join(_estimators_for(r)), axis=1)
_fits_all = len(spec_list) * 7
_fits_map = int(spec_list["estimators"].str.split().apply(lambda l: len([e for e in l if e != "ols"])).sum())
print(f"estimator map: {_fits_map} estimator x spec cells instead of {_fits_all} ({_fits_map / _fits_all:.0%})")
assert spec_list["spec"].is_unique, "duplicate spec names"

print(f"Total iterations: {len(spec_list)}")
print(spec_list["method"].value_counts().to_dict())
spec_list

# =========================================================
# Economist groups (config section 3, approved 2026-10-22): human1 "energy", human2 "activity".
# Same design as Jamaica's economist rows: PCA 3, two GDP lags, three-month flattening, all estimators.
# =========================================================
N = [dict({k: v for k, v in _g.items() if k != "label"}, size=len(_g["selected_vars"].split())) for _g in cfg.HUMAN_GROUPS]

for _h in N :
    spec_list = pd.concat([spec_list, pd.DataFrame([_h])], axis=0, ignore_index=True)

print("Human-spec tickets missing from the processed data:",
      {h['spec']: [t for t in h['selected_vars'].split() if t not in data.columns] for h in N})
spec_list


## 7. Training loop (vintages, leave-one-out refit, out-of-sample readings)


In [ ]:
# [tto:loop]
# =========================================================
# FULL JUPYTER-SAFE PARALLEL VERSION (Windows + Jupyter Lab)
# Parallelize across spec_list rows using joblib (loky)
# Save results EVERY 25 specs and at the end
# WITH A REAL PROGRESS BAR (completed specs + ETA)
# =========================================================
# The beta13/14 extensions all live inside the worker and are all switched from
# config.py section 6. With those left at their defaults this cell does exactly
# what it did on 2026-08-01, EXCEPT for PCA specs (2026-08-06): the scaler and
# PCA are now fitted on the training window only and applied to the test and
# OOS frames, instead of being refit on each test frame (which leaked the test
# window and made the components incomparable across fits). PCA-spec numbers
# are therefore not comparable with earlier runs; non-PCA specs are unchanged.

import traceback, timeit, os, warnings
warnings.filterwarnings("ignore")
import pandas as pd

from joblib import Parallel, delayed
from tqdm.auto import tqdm

# ---------------------------------------------------------
# 0) Thread caps (IMPORTANT) — set BEFORE heavy numeric work
# ---------------------------------------------------------
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
torch.set_num_threads(2)  # lets each worker use 2 CPU threads for PyTorch ops

# 8 cores -> typically best is 7 workers (leave 1 core free)
# If run['lstm']==1 frequently, set this to 4–6 instead.
MAX_WORKERS = cfg.MAX_WORKERS
_iter_lstm, _epoch_lstm = cfg.ITER_LSTM, cfg.EPOCH_LSTM   # 20 networks x 100 epochs (Jamaica)

print(f"Detected {os.cpu_count()} logical cores. Using {MAX_WORKERS} workers")

# ---------------------------------------------------------
# 1) Load existing output tables once
# ---------------------------------------------------------
start = timeit.default_timer()

try:
    stop
    result_table = pd.read_excel(f"{PATH_OUTPUT}/{forecast_file}", index_col="date")
    performance_table = pd.read_excel(f"{PATH_OUTPUT}/{performance_file}")
    outofsample_table = pd.read_excel(f"{PATH_OUTPUT}/{outofsample_file}", index_col="date")
    print("Existing files loaded.")
except Exception:
    print("No former files exist. Starting from spec 1.")
    performance_table = pd.DataFrame()
    result_table = pd.DataFrame()
    outofsample_table = pd.DataFrame()

print(f"Running and saving in: {performance_file}")


# ---------------------------------------------------------
# 2) Build list of specs to run (skip already-done)
# ---------------------------------------------------------
done_specs = set(performance_table["spec"].unique()) if "spec" in performance_table.columns else set()

# Everything the worker needs from config.py section 6, in one dict so the
# worker signature stays readable. Empty / None values switch each feature off.
opts = dict(                                   # loop knobs from config section 6 (Jamaica's values)
    fill               = cfg.NWCST_FILL,                     # "ffill"
    weight_decay       = cfg.SAMPLE_WEIGHT_DECAY,            # 0.95
    seed               = cfg.NWCST_SEED,                     # 42; the seed runner overrides per seed
    scale_linear       = bool(cfg.SCALE_LINEAR),
    ridge_alphas       = list(cfg.RIDGE_ALPHAS),
    winsor_z           = cfg.SHOCK_WINSOR_Z,                 # shock features are always on in this pipeline
    loo_dates          = [pd.to_datetime(x) for x in cfg.LOO_SHOCK_DATES],
    # GDP-lag features (2026-10-25): mode, publication lag, the pseudo-prints, the chain estimates and the data cut of the live frame
    gdp_lag_mode       = cfg.GDP_LAG_MODE,
    gdp_pub_lag        = int(metadata.set_index("ticket").loc[target_variable, "months_lag"]),
    pseudo_prints      = dict(cfg.PSEUDO_PRINTS),
    chain_estimates    = {},
    data_cut           = data.index[data.drop(columns=[target_variable]).notna().any(axis=1)].max(),
)
if cfg.GDP_LAG_MODE == "chain":
    # the base run's regime-aware nowcasts (backtest, every vintage) and live readings feed the chain
    _src = os.path.join(os.path.dirname(PATH_OUTPUT), cfg.CHAIN_SOURCE_RUN); _sds = cfg.CHAIN_SOURCE_RUN.split("_")[0]
    _ra = pd.read_excel(f"{_src}/{_sds} 4_rslt regime_aware_nowcast.xlsx", sheet_name="nowcast", parse_dates=["date"])
    _ce = {(pd.Timestamp(d), int(v)): float(x) for d, v, x in zip(_ra["date"], _ra["Vintage"], _ra["regime_aware_nowcast"])}
    _lp = f"{_src}/{_sds} 4_rslt live_quarter.xlsx"
    if os.path.exists(_lp):
        _lv = pd.read_excel(_lp, sheet_name="nowcast", parse_dates=["date"])
        _ce.update({(pd.Timestamp(d), int(v)): float(x) for d, v, x in zip(_lv["date"], _lv["Vintage"], _lv["regime_aware_nowcast"])})
    opts["chain_estimates"] = _ce
    print(f"[chain] {len(_ce)} (quarter, vintage) nowcasts loaded from {cfg.CHAIN_SOURCE_RUN} as lag estimates")
# rolling selections are attached per spec row (its own screen's ranking), see the rows loop below

rows = []
for _, row in spec_list.iterrows():
    spec = row["spec"]
    if spec in done_specs:
        continue
    d = row.to_dict()
    # Jamaica beta16: attach the rolling selections of THIS spec's screen, so the worker switches variables per test date
    _m = d.get("method")
    d["rolling_selections"] = rolling_selections.get(_m, {}) if isinstance(_m, str) else {}   # common/shared/human rows are static
    rows.append(d)

print(f"Total specs queued: {len(rows)} (skipping {len(done_specs)} already done)")


# ---------------------------------------------------------
# 3) Worker: runs ONE spec and returns its outputs
# ---------------------------------------------------------
def _run_one_spec_worker(row_dict, opts,
                         data, metadata,
                         target_variable, lags, run,
                         train_start_date, test_start_date, test_end_date,
                         _iter_, _iter_lstm, _epoch_lstm, desired_date):
    """
    Returns:
      (spec, perf_df, fcst_df, oos_df, loo_rows, err_str)
    where err_str is None if success, else traceback string.
    """
    try:
        # torch + loky deadlock/segfault if a worker spawns its own thread pool.
        try:
            import torch as _torch
            _torch.set_num_threads(1)
            _torch.set_num_interop_threads(1)
        except Exception:
            pass

        import numpy as _np

        spec = row_dict["spec"]
        XVars = row_dict["selected_vars"].split(" ")

        # ---- config.py section 6 knobs ----
        _rolling_sel = row_dict.get("rolling_selections") or {}   # per screen (Jamaica beta16)
        _fill_fn     = ffill_dataset if opts.get("fill") == "ffill" else mean_fill_dataset
        _decay       = opts.get("weight_decay")
        _seed_w      = opts.get("seed")
        _scale       = bool(opts.get("scale_linear"))
        _alphas      = opts.get("ridge_alphas")
        _wz          = opts.get("winsor_z")
        _loo_dates_w = opts.get("loo_dates") or []
        # GDP-lag features (2026-10-25): built from the real prints with the publication rule, see nwcst_helpers.gdp_lag_features
        _y_true   = data[target_variable].dropna()
        _lag_mode = opts.get("gdp_lag_mode", "vintage"); _pub = int(opts.get("gdp_pub_lag", 6))
        _pseudo   = opts.get("pseudo_prints") or {}; _chain = opts.get("chain_estimates") or {}

        # Rolling LARS: K comes off the spec row; the union of every date's top-K
        # is sliced out of `data` up front so each date's subset is available.
        _K_rolling = None
        if _rolling_sel and row_dict.get("rolling_enabled"):
            _K_raw = row_dict.get("K")
            try:
                _K_rolling = int(_K_raw) if _K_raw is not None and not pd.isna(_K_raw) else None
            except (ValueError, TypeError):
                _K_rolling = None
        if _rolling_sel and _K_rolling:
            _uni = set()
            for _lst in _rolling_sel.values():
                _uni.update(_lst[:_K_rolling])
            XVars = sorted(_uni)

        # params
        gdp_lags = row_dict.get("gdp_lags", 0)   # spec_list column is "gdp_lags"
        # Jamaica beta16: per-spec estimator map (spec_list column "estimators"); missing/empty -> all enabled estimators
        _allowed = row_dict.get("estimators")
        if isinstance(_allowed, str) and _allowed.strip():
            _allowed = set(_allowed.split())
            run = {k: (v if k in _allowed else 0) for k, v in run.items()}
        param_pca = row_dict.get("PCA", 3)
        data_flat = row_dict.get("data_flat", 3)

        dt_n = row_dict.get("dt_n", _iter_)
        rf_n = row_dict.get("rf_n", dt_n)
        gbt_n = row_dict.get("gbt_n", dt_n)

        # local imports inside worker (safe in spawned processes)
        from sklearn.preprocessing import StandardScaler
        from sklearn.decomposition import PCA

        scaler = StandardScaler()

        # -------------------------------------------------------------
        # Winsorizing helpers. Shock columns (the 9-character "…N"/"…R"
        # tickets and their flattened lags) are unbounded by construction, and a
        # single hurricane quarter can dominate a least-squares fit. Trees and
        # the LSTM are untouched -- an extreme value is information to them.
        # _wz is None whenever SHOCK_FEATURES = 0, and everything below no-ops.
        # -------------------------------------------------------------
        _shock_base = set()
        if _wz is not None:
            _shock_base = {c for c in data.columns
                           if isinstance(c, str) and len(c) == 9 and c[-1:] in ('N', 'R')}

        def _wins_bounds(df, cols):
            out = {}
            for c in cols:
                if c not in df.columns:
                    continue
                mu, sd = df[c].mean(), df[c].std()
                if sd is not None and pd.notna(sd) and sd > 0:
                    out[c] = (mu - _wz * sd, mu + _wz * sd)
            return out

        def _wins_apply(df, bounds):
            if not bounds:
                return df
            df = df.copy()
            for c, (lo, hi) in bounds.items():
                if c in df.columns:
                    df[c] = df[c].clip(lo, hi)
            return df

        # slice data for this spec
        df = data[[c for c in data.columns if c in XVars or c == target_variable]].copy()

        # GENERATE TRAIN/TEST DATASET
        df = df.reset_index(drop=False)
        df.dropna(axis=1, thresh=23, inplace=True)

        test = df.loc[(df.date >= train_start_date) & (df.date <= test_end_date), :].reset_index(drop=True)

        # DATES AND ACTUAL VALUES
        dates = pd.date_range(test_start_date, test_end_date, freq="3MS").strftime("%Y-%m-%d").tolist()
        actuals = list(test.loc[test.date.astype(str).isin(dates), target_variable].values)

        pred_ols  = {k: [] for k in lags}
        pred_olsr = {k: [] for k in lags}
        pred_enet = {k: [] for k in lags}
        pred_lasso= {k: [] for k in lags}
        pred_dt   = {k: [] for k in lags}
        pred_dt4  = {k: [] for k in lags}      # Jamaica beta16: single tree with 4-quarter leaves
        pred_rf   = {k: [] for k in lags}
        pred_gbt  = {k: [] for k in lags}
        pred_lstm = {k: [] for k in lags}
        _ttrain_loo_base = None               # Jamaica 2026-10-19: first test quarter's training table, kept for the leave-one-out refits

        for date in dates:

            # Rolling LARS: use the ranking computed at the latest test quarter
            # at or before this one -- never a later one.
            if _rolling_sel and _K_rolling:
                _td = pd.to_datetime(date)
                _avail = [t for t in _rolling_sel.keys() if t <= _td]
                _use_t = max(_avail) if _avail else min(_rolling_sel.keys())
                XVars_date = list(_rolling_sel[_use_t][:_K_rolling])
            else:
                XVars_date = XVars

            # Restrict the WHOLE frame -- training and test alike -- to this
            # quarter's variables. Subsetting only the training side would leave
            # the LSTM predicting on columns it never saw: nowcast_lstm iterates
            # the frame it is handed and looks each column up in the fill table
            # it stored at training time, so a surplus column raises KeyError
            # rather than being ignored. Every other estimator survives that
            # mismatch because predict_*() does X[train_vars] first.
            test_d = test
            if _rolling_sel and _K_rolling:
                test_d = test[[c for c in test.columns
                               if c in XVars_date or c == target_variable or c == "date"]]

            # define rolling-window training set
            train = test_d.loc[
                test_d.date <= str(pd.to_datetime(date) - pd.tseries.offsets.DateOffset(months=3))[:10],
                :
            ]
            transformed_train = _fill_fn(train, train)

            # -----------------------------------------------------------------
            # PCA scaling (if enabled) -- 2026-08-06 fix: fit the scaler and the
            # PCA ONCE on this quarter's training window and KEEP them (pca_q,
            # pca_cols), so the test rows in the lag loop and the OOS frame are
            # transformed into the SAME component space the models are trained
            # in. The old code refit both on each test frame, which leaked the
            # test window and made the components incomparable to the training
            # ones (PCA signs/order can flip between fits).
            if param_pca not in (False, 0, None):
                ncomp = 1 if param_pca is True else int(param_pca)

                X = transformed_train.set_index("date").drop(target_variable, axis=1)
                X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns).dropna(axis=1)

                pca_q = PCA(n_components=ncomp).fit(X)
                pca_cols = X.columns
                Z = pd.DataFrame(pca_q.transform(X)).rename(columns=lambda x: f"PC_{x}")

                transformed_train = pd.concat([transformed_train[["date", target_variable]], Z], axis=1)

            ttrain = flatten_data(transformed_train, target_variable, data_flat)

            # keep quarterly observations and drop early observations
            # NOTE: the mean is taken BEFORE the quarter filter, as it always has
            # been -- the original chained .fillna(ttrain.mean()) evaluated its
            # argument against the unfiltered frame. Do not fold this back into
            # the chain; it would change every stored number.
            _ttrain_mean = ttrain.mean(numeric_only=True)
            ttrain = ttrain.loc[ttrain.date.dt.month.astype(int).isin([3, 6, 9, 12]), :]
            ttrain = (ttrain.ffill().bfill() if opts.get("fill") == "ffill"
                      else ttrain.fillna(_ttrain_mean))
            ttrain = ttrain.dropna(axis=1).reset_index(drop=True)
            ttrain = gdp_lag_features(ttrain, gdp_lags, _y_true, _pub, _lag_mode)                        # training convention
            if _ttrain_loo_base is None:
                _ttrain_loo_base = ttrain.copy()    # the refit uses data to the quarter before the test window: real time for every test quarter

            xt = ttrain.drop(["date", target_variable], axis=1)
            yt = ttrain[target_variable]

            # winsorized copy for the linear models only
            _shock_cols = [c for c in xt.columns
                           if c in _shock_base or any(c.startswith(b + '_') for b in _shock_base)]
            _wb = _wins_bounds(xt, _shock_cols) if _shock_cols else {}
            xt_lin = _wins_apply(xt, _wb)

            # Exponential decay: a quarter `a` quarters old weighs decay**a, so
            # the fit leans on the recent past without discarding the distant one.
            if _decay is not None:
                _ages = (ttrain['date'].max() - ttrain['date']).dt.days.values / 90.0
                sample_weight = _np.power(float(_decay), _ages)
            else:
                sample_weight = None

            # -----------------------------------------------------------------
            # Apply model in train
            if run.get("ols", 0) == 1:
                ols_fit, ols_vars = fit_ols(yt, xt_lin, sample_weight=sample_weight)

            if run.get("olsr", 0) == 1:
                olsr_fit, olsr_vars = fit_olsr(yt, xt_lin, alphas=_alphas,
                                               sample_weight=sample_weight, scale=_scale)

            if run.get("enet", 0) == 1:
                enet_fit, enet_vars = fit_enet(yt, xt_lin, sample_weight=sample_weight, scale=_scale)

            if run.get("lasso", 0) == 1:
                lasso_fit, lasso_vars = fit_lasso(yt, xt_lin, sample_weight=sample_weight, scale=_scale)

            if run.get("dt", 0) == 1:
                dt_fit, dt_vars = fit_dt(yt, xt, ModelN=dt_n, sample_weight=sample_weight, seed=_seed_w)
            if run.get("dt4", 0) == 1:
                dt4_fit, dt4_vars = fit_dt(yt, xt, ModelN=dt_n, sample_weight=sample_weight, seed=_seed_w, min_leaf=4)

            if run.get("rf", 0) == 1:
                rf_fit, rf_vars = fit_rf(yt, xt, ModelN=rf_n, sample_weight=sample_weight, seed=_seed_w)

            if run.get("gbt", 0) == 1:
                gbt_fit, gbt_vars = fit_gbt(yt, xt, ModelN=gbt_n, sample_weight=sample_weight, seed=_seed_w)

            if run.get("lstm", 0) == 1:
                lstm_fit, lstm_vars = fit_lstm(transformed_train, gdp_lags=gdp_lags,
                                               n_models=_iter_lstm, train_episodes=_epoch_lstm,
                                               seed=_seed_w)

            # -----------------------------------------------------------------
            # Publication Lags in Test data
            for lag in lags:
                tmp_data = gen_lagged_data(metadata, test_d, date, lag)
                tmp_data = _fill_fn(train, tmp_data)

                if param_pca not in (False, 0, None):
                    # transform with the TRAIN-fitted scaler/PCA -- never refit
                    # on the test frame (see the note at the training block)
                    X = tmp_data.set_index("date").drop(target_variable, axis=1)
                    X = pd.DataFrame(scaler.transform(X), columns=X.columns)[pca_cols]
                    Z = pd.DataFrame(pca_q.transform(X)).rename(columns=lambda x: f"PC_{x}")

                    tmp_data = pd.concat([tmp_data[["date", target_variable]], Z], axis=1)

                test_data = flatten_data(tmp_data, target_variable, data_flat)
                test_data = gdp_lag_features(test_data, gdp_lags, _y_true, _pub, _lag_mode,
                                             reading_date=pd.to_datetime(date) + pd.DateOffset(months=int(lag)),
                                             estimates=_chain, pseudo=_pseudo)
                test_data = test_data.loc[test_data["date"] == date, :].drop(["date", target_variable], axis=1)
                test_data_lin = _wins_apply(test_data, _wb)

                # Predicted values
                if run.get("ols", 0) == 1:
                    pred_ols = predict_ols(ols_fit, test_data_lin, ols_vars, date, pred_ols, lag)

                if run.get("olsr", 0) == 1:
                    pred_olsr = predict_olsr(olsr_fit, test_data_lin, olsr_vars, date, pred_olsr, lag)

                if run.get("enet", 0) == 1:
                    pred_enet = predict_enet(enet_fit, test_data_lin, enet_vars, date, pred_enet, lag)

                if run.get("lasso", 0) == 1:
                    pred_lasso = predict_lasso(lasso_fit, test_data_lin, lasso_vars, date, pred_lasso, lag)

                if run.get("dt", 0) == 1:
                    pred_dt = predict_dt(dt_fit, test_data, dt_vars, date, pred_dt, lag)
                if run.get("dt4", 0) == 1:
                    pred_dt4 = predict_dt(dt4_fit, test_data, dt4_vars, date, pred_dt4, lag)

                if run.get("rf", 0) == 1:
                    pred_rf = predict_rf(rf_fit, test_data, rf_vars, date, pred_rf, lag)

                if run.get("gbt", 0) == 1:
                    pred_gbt = predict_gbt(gbt_fit, test_data, gbt_vars, date, pred_gbt, lag)

                if run.get("lstm", 0) == 1:
                    pred_lstm = predict_lstm(lstm_fit, tmp_data, lstm_vars, date, pred_lstm, lag, gdp_lags=gdp_lags)

        # -----------------------------------------------------------------
        # Leave-one-out shock residuals
        # -----------------------------------------------------------------
        # A shock quarter that sits INSIDE the training sample is fitted, not
        # forecast, so its in-sample residual says nothing about how the model
        # would have coped. Refitting with that one quarter held out and then
        # predicting it gives a genuine out-of-sample error for the one kind of
        # quarter we have almost no test observations of.
        loo_rows = []
        _tl = _ttrain_loo_base if _ttrain_loo_base is not None else ttrain   # Jamaica 2026-10-19: real-time table (the kit used the last quarter's)
        for _loo_date in _loo_dates_w:
            try:
                if "date" not in _tl.columns or not (_tl["date"] == _loo_date).any():
                    continue
                _mask      = _tl["date"] == _loo_date
                _shock_row = _tl[_mask].reset_index(drop=True)
                _train_loo = _tl[~_mask].reset_index(drop=True)

                _yt_loo   = _train_loo[target_variable]
                _xt_loo   = _train_loo.drop(["date", target_variable], axis=1)
                _xt_shock = _shock_row.drop(["date", target_variable], axis=1)
                _y_actual = float(_shock_row[target_variable].iloc[0])

                # Re-derive the winsorizing bounds and the weights on the held-out
                # sample, so nothing from the excluded quarter leaks back in.
                _wb_loo       = _wins_bounds(_xt_loo, _shock_cols) if _shock_cols else {}
                _xt_lin_loo   = _wins_apply(_xt_loo,   _wb_loo)
                _xt_lin_shock = _wins_apply(_xt_shock, _wb_loo)

                if _decay is not None:
                    _ages_loo = (_train_loo["date"].max() - _train_loo["date"]).dt.days / 90.0
                    _sw_loo = _np.power(float(_decay), _ages_loo).values
                else:
                    _sw_loo = None

                def _loo_predict(fit_obj, x_df):
                    try:
                        if isinstance(fit_obj, list):      # dt / rf / gbt return a bag
                            preds = []
                            for _m in fit_obj:
                                try:
                                    preds.append(float(_m.predict(x_df)[0]))
                                except Exception:
                                    pass
                            return float(_np.nanmean(preds)) if preds else None
                        return float(fit_obj.predict(x_df)[0])
                    except Exception:
                        return None

                _loo_models = [
                    ("ols",   "OLS",   lambda: fit_ols(_yt_loo, _xt_lin_loo, sample_weight=_sw_loo),   True),
                    ("olsr",  "OLSR",  lambda: fit_olsr(_yt_loo, _xt_lin_loo, alphas=_alphas,
                                                        sample_weight=_sw_loo, scale=_scale),          True),
                    ("enet",  "ENET",  lambda: fit_enet(_yt_loo, _xt_lin_loo, sample_weight=_sw_loo,
                                                        scale=_scale),                                 True),
                    ("lasso", "LASSO", lambda: fit_lasso(_yt_loo, _xt_lin_loo, sample_weight=_sw_loo,
                                                         scale=_scale),                                True),
                    ("dt",    "DT",    lambda: fit_dt(_yt_loo, _xt_loo, ModelN=dt_n,
                                                      sample_weight=_sw_loo, seed=_seed_w),            False),
                    ("dt4",   "DT4",   lambda: fit_dt(_yt_loo, _xt_loo, ModelN=dt_n,
                                                      sample_weight=_sw_loo, seed=_seed_w, min_leaf=4), False),
                    ("rf",    "RF",    lambda: fit_rf(_yt_loo, _xt_loo, ModelN=rf_n,
                                                      sample_weight=_sw_loo, seed=_seed_w),            False),
                    ("gbt",   "GBT",   lambda: fit_gbt(_yt_loo, _xt_loo, ModelN=gbt_n,
                                                       sample_weight=_sw_loo, seed=_seed_w),           False),
                ]
                for _key, _label, _fitter, _is_linear in _loo_models:
                    if run.get(_key, 0) != 1:
                        continue
                    _fit, _ = _fitter()
                    _pred = _loo_predict(_fit, _xt_lin_shock if _is_linear else _xt_shock)
                    if _pred is not None:
                        loo_rows.append({"spec": spec, "estimator": _label,
                                         "shock_date": str(pd.to_datetime(_loo_date).date()),
                                         "actual": _y_actual, "predicted": _pred,
                                         "residual": _pred - _y_actual})
            except Exception as _loo_err:
                print(f"[LOO WARNING spec={spec} date={_loo_date}] {_loo_err}")

        # -----------------------------------------------------------------
        # Prepare Tables (ONLY this spec)
        perf_parts = []
        fcst_parts = []

        if run.get("ols", 0) == 1:
            perf_parts.append(perform_tab(pred_ols, "OLS", spec, actuals, lags))
            fcst_parts.append(forecast_table(pred_ols, "OLS", spec, actuals, dates))

        if run.get("olsr", 0) == 1:
            perf_parts.append(perform_tab(pred_olsr, "OLSR", spec, actuals, lags))
            fcst_parts.append(forecast_table(pred_olsr, "OLSR", spec, actuals, dates))

        if run.get("enet", 0) == 1:
            perf_parts.append(perform_tab(pred_enet, "ENET", spec, actuals, lags))
            fcst_parts.append(forecast_table(pred_enet, "ENET", spec, actuals, dates))

        if run.get("lasso", 0) == 1:
            perf_parts.append(perform_tab(pred_lasso, "LASSO", spec, actuals, lags))
            fcst_parts.append(forecast_table(pred_lasso, "LASSO", spec, actuals, dates))

        if run.get("dt", 0) == 1:
            perf_parts.append(perform_tab(pred_dt, "DT", spec, actuals, lags))
            fcst_parts.append(forecast_table(pred_dt, "DT", spec, actuals, dates))
        if run.get("dt4", 0) == 1:
            perf_parts.append(perform_tab(pred_dt4, "DT4", spec, actuals, lags))
            fcst_parts.append(forecast_table(pred_dt4, "DT4", spec, actuals, dates))

        if run.get("rf", 0) == 1:
            perf_parts.append(perform_tab(pred_rf, "RF", spec, actuals, lags))
            fcst_parts.append(forecast_table(pred_rf, "RF", spec, actuals, dates))

        if run.get("gbt", 0) == 1:
            perf_parts.append(perform_tab(pred_gbt, "GBT", spec, actuals, lags))
            fcst_parts.append(forecast_table(pred_gbt, "GBT", spec, actuals, dates))

        if run.get("lstm", 0) == 1:
            perf_parts.append(perform_tab(pred_lstm, "LSTM", spec, actuals, lags))
            fcst_parts.append(forecast_table(pred_lstm, "LSTM", spec, actuals, dates))

        perf_df = pd.concat(perf_parts, axis=0) if perf_parts else pd.DataFrame()
        fcst_df = pd.concat(fcst_parts, axis=0) if fcst_parts else pd.DataFrame()

        # -----------------------------------------------------------------
        # OUT OF SAMPLE ESTIMATION
        # -----------------------------------------------------------------
        oos_df = pd.DataFrame()      # stays empty when desired_date is None
        if desired_date is not None:
            # Same column-parity requirement as the test loop above: the fitted
            # models are the ones from the LAST test quarter, so the OOS frame
            # must carry that quarter's variables and no others.
            if _rolling_sel and _K_rolling:
                df = df[[c for c in df.columns
                         if c in XVars_date or c == target_variable or c == "date"]]

            # Extend dataframe to desired_date
            while desired_date > df['date'].max():
                df.loc[len(df), "date"] = df['date'].max() + pd.DateOffset(months=1)

            transformed_new_data = _fill_fn(df, df)

            # Apply PCA if specified -- transform with the scaler/PCA fitted on
            # the LAST test quarter's training window: the same component space
            # the fitted models expect (never refit on the OOS frame)
            if param_pca not in (False, 0, None):
                X = transformed_new_data.set_index('date').drop(target_variable, axis=1)
                X = pd.DataFrame(scaler.transform(X), columns=X.columns)[pca_cols]
                Z = pd.DataFrame(pca_q.transform(X)).rename(columns=lambda x: f'PC_{x}')

                transformed_new_data = pd.concat([transformed_new_data[['date', target_variable]], Z], axis=1)

            # Prepare OOS data
            transformed_data = flatten_data(transformed_new_data, target_variable, data_flat)
            transformed_data = gdp_lag_features(transformed_data, gdp_lags, _y_true, _pub, _lag_mode,
                                                reading_date=opts.get("data_cut"), estimates=_chain, pseudo=_pseudo)
            transformed_data = transformed_data[transformed_data['date'] > date]
            transformed_data = transformed_data[transformed_data['date'] <= desired_date]
            transformed_data = transformed_data.loc[transformed_data.date.dt.month.astype(int).isin([3,6,9,12]),:]
            # transformed_train.mean() -- the TRAINING frame from the last test
            # quarter, and called without numeric_only, exactly as before.
            transformed_data = (transformed_data.ffill().bfill() if opts.get("fill") == "ffill"
                                else transformed_data.fillna(transformed_train.mean()))
            transformed_data = transformed_data.dropna(axis=1)
            new_dates = transformed_data['date']
            transformed_data = transformed_data.drop(["date", target_variable], axis=1).reset_index(drop=True)
            transformed_data_lin = _wins_apply(transformed_data, _wb)

            # Initialize empty outofsample_table (assuming it's a list or DataFrame)
            outofsample_table = pd.DataFrame()

            # Run OOS forecasts
            if run.get('ols', 0) == 1:
                outofsample_table = oos_ols(ols_fit, transformed_data_lin, train_vars=ols_vars,
                                           table=outofsample_table, new_dates=new_dates, spec=spec, desired_date=desired_date)
            if run.get('olsr', 0) == 1:
                outofsample_table = oos_olsr(olsr_fit, transformed_data_lin, train_vars=olsr_vars,
                                            table=outofsample_table, new_dates=new_dates, spec=spec, desired_date=desired_date)
            if run.get('enet', 0) == 1:
                outofsample_table = oos_enet(enet_fit, transformed_data_lin, train_vars=enet_vars,
                                            table=outofsample_table, new_dates=new_dates, spec=spec, desired_date=desired_date)
            if run.get('lasso', 0) == 1:
                outofsample_table = oos_lasso(lasso_fit, transformed_data_lin, train_vars=lasso_vars,
                                             table=outofsample_table, new_dates=new_dates, spec=spec, desired_date=desired_date )
            if run.get('dt', 0) == 1:
                outofsample_table = oos_dt(dt_fit, transformed_data, train_vars=dt_vars,
                                          table=outofsample_table, new_dates=new_dates, spec=spec, desired_date=desired_date )
            if run.get('dt4', 0) == 1:
                outofsample_table = oos_dt(dt4_fit, transformed_data, train_vars=dt4_vars,
                                          table=outofsample_table, new_dates=new_dates, spec=spec, desired_date=desired_date, label='DT4')   # 2026-10-26 label fix
            if run.get('rf', 0) == 1:
                outofsample_table = oos_rf(rf_fit, transformed_data, train_vars=rf_vars,
                                          table=outofsample_table, new_dates=new_dates, spec=spec, desired_date=desired_date )
            if run.get('gbt', 0) == 1:
                outofsample_table = oos_gbt(gbt_fit, transformed_data, train_vars=gbt_vars,
                                           table=outofsample_table, new_dates=new_dates , spec=spec, desired_date=desired_date)
            if run.get('lstm', 0) == 1:
                outofsample_table = oos_lstm(lstm_fit, transformed_new_data, train_vars=lstm_vars,
                                            table=outofsample_table, new_dates=new_dates, spec=spec, desired_date=desired_date)

            # Convert to DataFrame if needed
            oos_df = pd.DataFrame(outofsample_table) if isinstance(outofsample_table, list) else outofsample_table

        return spec, perf_df, fcst_df, oos_df, loo_rows, None

    except Exception:
        return row_dict.get("spec", "UNKNOWN"), None, None, None, None, traceback.format_exc()



# ---------------------------------------------------------
# 4) Run in parallel (joblib loky) and collect results
# ---------------------------------------------------------
perf_out, fcst_out, oos_out, loo_out = [], [], [], []
failed = []

# NEW: Set batch size for incremental saves
SAVE_BATCH_SIZE = cfg.SAVE_BATCH_SIZE
specs_processed = 0

if len(rows) > 0:
    results = []
    with tqdm(total=len(rows), desc="Processing specs") as pbar:
        for result in Parallel(n_jobs=MAX_WORKERS, backend="loky", verbose=5, return_as="generator")(
            delayed(_run_one_spec_worker)(
                row_dict, opts, data, metadata, target_variable, lags, run,
                train_start_date, test_start_date, test_end_date,
                _iter_, _iter_lstm, _epoch_lstm, desired_date
            ) for row_dict in rows
        ):
            results.append(result)
            pbar.update(1)

    for spec, perf_df, fcst_df, oos_df, loo_list, err in results:
        if err is not None:
            failed.append(spec)
            print(f"FAILED spec={spec}\n{err}\n")
            continue

        if perf_df is not None and not perf_df.empty:
            perf_out.append(perf_df)
        if fcst_df is not None and not fcst_df.empty:
            fcst_out.append(fcst_df)
        if oos_df is not None and not oos_df.empty:
            oos_out.append(oos_df)
        if loo_list:
            loo_out.extend(loo_list)

        specs_processed += 1

        # NEW: Save every SAVE_BATCH_SIZE specs
        if specs_processed % SAVE_BATCH_SIZE == 0:
            print(f"\n{'='*60}")
            print(f"SAVING BACKUP after {specs_processed} specs")
            print(f"{'='*60}")
            # Merge with existing tables
            if perf_out:
                temp_perf = pd.concat([performance_table] + perf_out, axis=0)
                temp_perf.to_excel(f"{PATH_OUTPUT}/{performance_file}")
                print(f"Saved {len(temp_perf)} rows to {performance_file}")

            if fcst_out:
                temp_fcst = pd.concat([result_table] + fcst_out, axis=0)
                temp_fcst.index = pd.to_datetime(temp_fcst.index).strftime("%Y-%m-%d")
                temp_fcst.to_excel(f"{PATH_OUTPUT}/{forecast_file}", index_label="date")
                print(f"Saved {len(temp_fcst)} rows to {forecast_file}")

            if oos_out:
                temp_oos = pd.concat([outofsample_table] + oos_out, axis=0)
                temp_oos.index = pd.to_datetime(temp_oos.index).strftime("%Y-%m-%d")
                temp_oos.to_excel(f"{PATH_OUTPUT}/{outofsample_file}", index_label="date")
                print(f"Saved {len(temp_oos)} rows to {outofsample_file}")

            print(f"{'='*60}\n")

# ---------------------------------------------------------
# 5) Final merge and save (FAST)
# ---------------------------------------------------------
if perf_out:
    performance_table = pd.concat([performance_table] + perf_out, axis=0, ignore_index=False)

if fcst_out:
    result_table = pd.concat([result_table] + fcst_out, axis=0)

if oos_out:
    outofsample_table = pd.concat([outofsample_table] + oos_out, axis=0)

# Final save
performance_table.to_excel(f"{PATH_OUTPUT}/{performance_file}")

result_table.index = pd.to_datetime(result_table.index).strftime("%Y-%m-%d")
result_table.index = pd.to_datetime(result_table.index)
result_table.to_excel(f"{PATH_OUTPUT}/{forecast_file}", index_label="date")

outofsample_table.index = pd.to_datetime(outofsample_table.index).strftime("%Y-%m-%d")
outofsample_table.index = pd.to_datetime(outofsample_table.index)
outofsample_table.to_excel(f"{PATH_OUTPUT}/{outofsample_file}", index_label="date")

# Leave-one-out shock residuals (empty unless LOO_SHOCK_DATES is set). The
# watchlist cell reads this file for its shock-robustness panel.
shock_residual_file = f"{date_str} 3_nwcst Tab shock_residuals.xlsx"
if loo_out:
    shock_residuals = pd.DataFrame(loo_out)
    shock_residuals.to_excel(f"{PATH_OUTPUT}/{shock_residual_file}", index=False)
    print(f"Wrote {len(shock_residuals)} leave-one-out shock residuals to {shock_residual_file}")

run_time = timeit.default_timer() - start
print(f"\nTotal runtime: {run_time:.2f} seconds")
print(f"Saved in {performance_file}")
if failed:
    print(f"Failed specs ({len(failed)}): {failed}")

print(performance_table['RMSE'].describe())
print(f"Original Benchmark: {benchmark}")
performance_table


## 8. Watchlist, regime-aware nowcast, dispersion bands


In [ ]:
# [tto:post]
# =========================================================
# beta15 watchlist (clusters re-identified on the beta15 pool, 2026-10-17; see the markdown above)
# Panel 1: P25/P50/P75 of the pooled readings per (date, Vintage)        [beta14 p75 bug fixed: was the mean]
# Panel 2: cluster medians vs the pooled median; alert = divergence <= -ALERT_PP (ALERT_SIDE below) or |divergence| >= ALERT_PP (both)
# Panel 3: shock-robustness score per (spec, estimator) from the COVID LOO residuals (unchanged)
# =========================================================
import pandas as pd, numpy as np, re

fcst_w = pd.read_excel(f"{PATH_OUTPUT}/{forecast_file}", parse_dates=["date"])
fcst_w = reading_guard(fcst_w[pool_mask(fcst_w)], label="watchlist")   # pool filter (cell 2) + reading guard (2026-10-25)
long_w = fcst_w.melt(id_vars=["date", "spec", "estimator", "actuals"],
                     value_vars=list(VINTAGES), var_name="horizon", value_name="pred")   # VINTAGES: cell 2 (any grid)
long_w["Vintage"] = long_w["horizon"].map(VINTAGES)
long_w = long_w.dropna(subset=["pred"])

# ---- Panel 1: pooled spread ----
spread = long_w.groupby(["date", "Vintage"]).agg(
    n=("pred", "size"),
    p25=("pred", lambda s: s.quantile(0.25)),
    p50=("pred", "median"),
    p75=("pred", lambda s: s.quantile(0.75)),
    actual=("actuals", "first"),
).reset_index().round(3)

# ---- Panel 2: clusters ----
# Spec names are <method><K><variant>; bench<K> is the base variant of the best method -> resolve from spec_list.
_bench_method = {r.spec: r.method for r in spec_list.itertuples() if str(r.spec).startswith("bench")}
def _family(s):
    m = re.match(r"^(f?(?:lars|sis|tstat|shap|ocmt|ibma)|common|shared|bench|human)(\d*)(flats|p5l2|p5|p3)?$", s)   # f-prefix: frozen groups (2026-10-25)
    if not m:
        return None, None, None
    meth = _bench_method.get(s, m.group(1)) if m.group(1) == "bench" else m.group(1)
    return meth, (int(m.group(2)) if m.group(2) else None), (m.group(3) or "base")

FACTOR, RAW = {"p3", "p5", "p5l2"}, {"base", "flats"}
# Cluster definitions live in config section 7 (WATCHLIST): empty until the identification run (step 7) fills them.
# name: (vintage or None = all, methods, variants, estimators or None = all, (K_from, K_to), note)
WATCHLIST = {k: (v["vintage"], set(v["methods"]), set(v["variants"]), (set(v["estimators"]) if v["estimators"] else None),
                 tuple(v["K"]), v["note"], bool(v.get("alerts", True))) for k, v in cfg.WATCHLIST.items()}
if not WATCHLIST:
    print("[watchlist] no clusters defined in config.WATCHLIST: the watchlist panel is empty; the regime can switch only on the "
          f"breadth trigger ({'enabled' if cfg.BREADTH_TRIGGER else 'disabled'})")
long_w["method"], long_w["K"], long_w["variant"] = zip(*long_w["spec"].apply(_family))
pooled_w = long_w.groupby(["date", "Vintage"])["pred"].median().rename("pooled_median")

panels, _summary = {}, []
for name, (V, meths, vars_, ests, (klo, khi), note, can_alert) in WATCHLIST.items():
    m = long_w["method"].isin(meths) & long_w["variant"].isin(vars_)
    if ests is not None:
        m &= long_w["estimator"].isin(ests)
    if klo is not None:
        m &= long_w["K"].between(klo, khi)
    if V is not None:
        m &= long_w["Vintage"].eq(V)
    g = long_w[m]
    if g.empty:
        print(f"[watchlist] {name}: no cells match -> skipped"); continue
    p = g.groupby(["date", "Vintage"]).agg(n=("pred", "size"),
                                           p25=("pred", lambda s: s.quantile(0.25)),
                                           median=("pred", "median"),
                                           p75=("pred", lambda s: s.quantile(0.75)),
                                           actual=("actuals", "first")).join(pooled_w)
    p["divergence"] = p["median"] - p["pooled_median"]
    # 2026-10-26 night: ALERT_SIDE "below" = crash alarm (cluster median at least ALERT_PP below the pool), "both" = either side
    # 2026-10-27 (user decision, option 3): flag = side of a divergence beyond ALERT_PP (-1 below, +1 above, 0 inside); only the
    # ALERT_SIDE side becomes an alert (the one thing that can switch the regime). Upward flags are information only.
    p["flag"] = np.where(p["divergence"] <= -cfg.ALERT_PP, -1, np.where(p["divergence"] >= cfg.ALERT_PP, 1, 0)) * int(can_alert)
    _div_hit = (p["flag"] == -1) if cfg.ALERT_SIDE == "below" else (p["flag"] != 0)
    p["alert"] = _div_hit.astype(int)
    p["cluster"] = name
    panels[name] = p.reset_index().round(3)
    _summary.append(p.reset_index().assign(cluster=name)[["date", "Vintage", "cluster", "median", "pooled_median", "divergence", "flag", "alert", "actual"]])
watch_summary = (pd.concat(_summary).sort_values(["date", "Vintage", "cluster"]).round(3) if _summary
                 else pd.DataFrame(columns=["date", "Vintage", "cluster", "median", "pooled_median", "divergence", "flag", "alert", "actual"]))
_alerts = watch_summary[watch_summary["alert"] == 1]
print(f"Watchlist alerts (cluster median - pooled median <= -{cfg.ALERT_PP:g} pp):" if cfg.ALERT_SIDE == "below" else f"Watchlist alerts (|cluster median - pooled median| >= {cfg.ALERT_PP:g} pp):")
print(_alerts.to_string(index=False) if len(_alerts) else "  none")
_flags_up = watch_summary[watch_summary["flag"] == 1]
print(f"Upward flags (cluster median - pooled median >= +{cfg.ALERT_PP:g} pp, information only, never switch the regime):")
print(_flags_up.to_string(index=False) if len(_flags_up) else "  none")

# ---- Panel 3: shock-robustness score (COVID LOO residuals) ----
shock_score = None
try:
    loo_w = pd.read_excel(f"{PATH_OUTPUT}/{date_str} 3_nwcst Tab shock_residuals.xlsx")
    shock_score = loo_w[["spec", "estimator", "residual"]].copy()
    shock_score["shock_score_abs"] = shock_score["residual"].abs()
    shock_score = shock_score.sort_values("shock_score_abs")
except Exception as e:
    print(f"[watchlist] no shock_residuals.xlsx ({e}) - shock-score panel skipped")

watch_out = f"{PATH_OUTPUT}/{date_str} 4_rslt watchlist.xlsx"
with pd.ExcelWriter(watch_out) as xw:
    spread.to_excel(xw, sheet_name="spread", index=False)
    watch_summary.to_excel(xw, sheet_name="watchlist_summary", index=False)
    for name, p in panels.items():
        p.to_excel(xw, sheet_name=name[:31], index=False)
    pd.DataFrame([{"cluster": k, "vintage": v[0] if v[0] is not None else "all", "methods": ",".join(sorted(v[1])),
                   "variants": ",".join(sorted(v[2])), "estimators": ",".join(sorted(v[3])) if v[3] else "all",
                   "K_from": v[4][0], "K_to": v[4][1], "note": v[5], "alerts": v[6]} for k, v in WATCHLIST.items()]
                 ).to_excel(xw, sheet_name="definitions", index=False)
    if shock_score is not None:
        shock_score.round(3).to_excel(xw, sheet_name="shock_robustness", index=False)
print(f"[OK] Wrote watchlist to {watch_out}")

# =========================================================
# beta15 regime-aware nowcast (2026-10-18): automatic, real-time regime; expanding-window weights
#  1. REGIME. A quarter q is in the shock regime at vintage v if, on the information set of that vintage,
#     the breadth rule fires in any observed month of q: >= max(4, SHOCK_SHARE * n) of the n monthly
#     variables carrying shock features (guard-cleaned) sit below -3 sigma of their training distribution.
#     Publication lags from the metadata decide which months are observed. No typed dates; an explicit,
#     logged override (REGIME_OVERRIDE) replaces the analyst's silent list.
#  2. WEIGHTS. Inverse squared RMSE per (spec, estimator, vintage), computed only on test quarters whose
#     official GDP print was available at the reading date (release lag GDP_RELEASE_LAG months):
#     ordinary weights from past ordinary quarters, shock weights from past shock quarters.
#  3. FALLBACKS, stated: no past ordinary residual yet -> equal weights. The shock record starts with the pandemic
#     leave-one-out residuals (2026-10-19), so a flagged quarter uses shock weights from the first shock on; only a
#     missing shock_residuals file sends it back to ordinary weights.
# =========================================================
import pandas as pd, numpy as np

GDP_RELEASE_LAG = int(metadata.set_index("ticket").loc[target_variable, "months_lag"])   # from the metadata (Trinidad: 6, see code/README_pipeline.md)
print(f"[regime] GDP release lag from metadata: {GDP_RELEASE_LAG} months")
BURN_IN_MIN_PRINTS = cfg.BURN_IN_MIN_PRINTS   # 3: ordinary weights need at least 3 published prints on record; before that every cell gets the same weight
REGIME_OVERRIDE = {"add": dict(cfg.REGIME_OVERRIDE_ADD), "remove": dict(cfg.REGIME_OVERRIDE_REMOVE)}   # analyst overrides, logged with their reason

fcst_ra = pd.read_excel(f"{PATH_OUTPUT}/{forecast_file}", parse_dates=["date"])
fcst_ra = reading_guard(fcst_ra[pool_mask(fcst_ra)], label="regime")   # pool filter (cell 2) + reading guard (2026-10-25)

# ---- pandemic on the shock record (2026-10-19): one leave-one-out residual per cell and seed for 2020-Q2 ----
try:
    _loo = pd.read_excel(f"{PATH_OUTPUT}/{date_str} 3_nwcst Tab shock_residuals.xlsx")
    _loo = _loo[pool_mask(_loo)][["spec", "estimator", "residual"]].rename(columns={"residual": "resid"})
    _cells = fcst_ra[["spec", "estimator"]].drop_duplicates()
    _missing = _cells.merge(_loo[["spec", "estimator"]].drop_duplicates(), how="left", indicator=True).query("_merge == 'left_only'")[["spec", "estimator"]]
    _loo_med = float(_loo.groupby(["spec", "estimator"])["resid"].apply(lambda x: np.sqrt(np.mean(x ** 2))).median())
    PANDEMIC_RECORD = pd.concat([_loo, _missing.assign(resid=_loo_med)], ignore_index=True)   # cells without a refit (LSTM) sit at the median
    print(f"[regime] pandemic record: {_loo[['spec', 'estimator']].drop_duplicates().shape[0]} cells from leave-one-out, "
          f"{len(_missing)} cells at the median residual {_loo_med:.1f}")
except FileNotFoundError:
    PANDEMIC_RECORD = None; print("[regime] no shock_residuals file: shock quarters fall back to ordinary weights")
# VINTAGES comes from cell 2 (any grid, 2026-10-25)
long_df = fcst_ra.melt(id_vars=["date", "spec", "estimator", "actuals"], value_vars=list(VINTAGES),
                       var_name="horizon", value_name="pred")
long_df["Vintage"] = long_df["horizon"].map(VINTAGES)
long_df = long_df.dropna(subset=["pred"])
long_df["resid"] = long_df["pred"] - long_df["actuals"]

# ---- 1. real-time breadth indicator per (quarter, vintage) ----
# Fixed panel: every guard-cleaned monthly candidate (not only the variables the screens selected), so the
# regime does not change with the selection. Standardisation against the training distribution of the
# QUARTERLY MINIMA, exactly as the breadth feature in cell 10.
_vars = [v for v in pre_selected_columns_monthly if v in data.columns and v not in broken_vars]
_lag = metadata.set_index("ticket")["months_lag"].reindex(_vars).fillna(1).astype(int)
_qmin_tr = data.loc[data.index < test_start_date, _vars].resample("QS").min()
_mu, _sg = _qmin_tr.mean(), _qmin_tr.std().replace(0, np.nan)
_thr = max(4, int(np.ceil(SHOCK_SHARE * len(_vars))))

def breadth_at(q_end, v):
    """Number of variables whose within-quarter minimum (over the months observed at vintage v) is below -3 sigma."""
    q_start = q_end - pd.DateOffset(months=2); reading = q_end + pd.DateOffset(months=v)
    cnt = 0
    for var in _vars:
        last_obs = reading - pd.DateOffset(months=int(_lag[var]))          # latest month published at the reading date
        obs = data.loc[q_start:min(q_end, last_obs), var].dropna()
        if len(obs) and (obs.min() - _mu[var]) / _sg[var] < -3:
            cnt += 1
    return cnt

test_q = sorted(long_df["date"].unique())
regime = pd.DataFrame([{"date": q, "Vintage": v, "breadth": breadth_at(pd.Timestamp(q), v)}
                       for q in test_q for v in VINTAGES.values()])
# ---- trigger (2026-10-18): breadth threshold, OR a watchlist alert corroborated by broken variables ----
# Trinidad 2026-10-25: BREADTH_TRIGGER = False in config, so only the corroborated watchlist alert switches the regime.
# The watchlist (previous cell) is specific and early for the kind of shock it was identified on (the hurricane:
# early cluster at V-1) and blind to the pandemic; the breadth count is generic but late (pandemic at V0 only,
# hurricane never above its threshold). Neither suffices alone, so the regime switches on either, and a
# watchlist alert counts only if at least MIN_BROKEN_FOR_ALERT variables in the panel are broken at that vintage.
# NOTE: the corroboration minimum was first set at a fixed 3 broken variables (the count behind the early
# cluster's hurricane alert at V-1 on Jamaica). It is written as max(3, floor(threshold/3)) so that it scales with the panel:
# 3 today (threshold 10 of 181), 5 with a 300-variable panel (threshold 15). Once triggered at a vintage, the quarter stays in the shock
# regime at the later vintages of that quarter ("first vintage at which").
MIN_BROKEN_FOR_ALERT = max(3, _thr // 3)
_wa = watch_summary[watch_summary["alert"] == 1][["date", "Vintage"]].drop_duplicates().assign(watch_alert=True)
regime = regime.merge(_wa, on=["date", "Vintage"], how="left"); regime["watch_alert"] = regime["watch_alert"].fillna(False).astype(bool)
regime["trig_breadth"] = (regime["breadth"] >= _thr) & cfg.BREADTH_TRIGGER   # Trinidad: False = corroboration only (2026-10-25)
regime["trig_watch"] = regime["watch_alert"] & (regime["breadth"] >= MIN_BROKEN_FOR_ALERT)
regime = regime.sort_values(["date", "Vintage"])
regime["is_shock"] = regime.groupby("date")[["trig_breadth", "trig_watch"]].transform("cummax").any(axis=1)
regime["trigger"] = np.select([regime.trig_breadth & regime.trig_watch, regime.trig_breadth, regime.trig_watch], ["both", "breadth", "watchlist"], "")
print(f"[regime] watchlist alerts need >= {MIN_BROKEN_FOR_ALERT} broken variables; triggers by (quarter, vintage):")
print(regime[regime["trigger"] != ""][["date", "Vintage", "breadth", "trigger"]].to_string(index=False) if (regime["trigger"] != "").any() else "  none")
for d_, why in REGIME_OVERRIDE["add"].items():    regime.loc[regime["date"] == pd.Timestamp(d_), "is_shock"] = True
for d_, why in REGIME_OVERRIDE["remove"].items(): regime.loc[regime["date"] == pd.Timestamp(d_), "is_shock"] = False
regime["override"] = regime["date"].dt.strftime("%Y-%m-%d").map({**REGIME_OVERRIDE["add"], **REGIME_OVERRIDE["remove"]}).fillna("")
regime["threshold"], regime["n_panel"] = _thr, len(_vars)      # read by the chart cells
print(f"[regime] breadth threshold {_thr} of {len(_vars)}; shock-regime (quarter, vintage) cells: {int(regime['is_shock'].sum())}")

# ---- 2. expanding-window weights ----
long_df = long_df.merge(regime[["date", "Vintage", "is_shock"]], on=["date", "Vintage"])
long_df["q_end"] = long_df["date"]                 # quarter-end month (dates are Mar/Jun/Sep/Dec)

def known_actual_cutoff(q_end, v):
    """Quarters whose official print is available at the reading date q_end + v months."""
    return q_end + pd.DateOffset(months=v) - pd.DateOffset(months=GDP_RELEASE_LAG)
regime["prints_known"] = [sum(pd.Timestamp(d) <= known_actual_cutoff(pd.Timestamp(q), v) for d in test_q)
                          for q, v in zip(regime["date"], regime["Vintage"])]      # prints behind the weights
regime["burn_in"] = regime["prints_known"] < BURN_IN_MIN_PRINTS                     # equal weights in these cells; the chart hatches them

def weights_for(q_end, v, shock):
    cut = known_actual_cutoff(q_end, v)
    hist = long_df[(long_df["Vintage"] == v) & (long_df["q_end"] <= cut) & (long_df["is_shock"] == shock)][["spec", "estimator", "resid"]]
    if not shock and sum(pd.Timestamp(d) <= cut for d in test_q) < BURN_IN_MIN_PRINTS:
        return None                                   # burn-in: a record of 1-2 prints is a lottery, not a ranking -> equal weights
    if shock and PANDEMIC_RECORD is not None:
        hist = pd.concat([hist, PANDEMIC_RECORD], ignore_index=True)    # the pandemic is on the record at every vintage
    if hist.empty:
        return None
    rmse = hist.groupby(["spec", "estimator"])["resid"].apply(lambda x: np.sqrt(np.nanmean(x ** 2)))
    w = (1.0 / rmse ** 2).replace([np.inf, -np.inf], np.nan).fillna(0)
    return w / w.sum() if w.sum() > 0 else None

out_rows, WEIGHT_CACHE = [], {}
for (q, v), g in long_df.groupby(["date", "Vintage"]):
    shock = bool(g["is_shock"].iloc[0]); q_end = pd.Timestamp(q)
    w = weights_for(q_end, v, shock=True) if shock else None
    shock_w_available = w is not None
    if w is None:
        w = weights_for(q_end, v, shock=False)          # ordinary weights (or shock regime without shock history)
    if w is None:                                        # first test quarter: no residual history at all
        pred = float(g["pred"].mean()); scheme = "equal"
    else:
        wk = np.array([w.get((s, e), 0.0) for s, e in zip(g["spec"], g["estimator"])])
        pred = float((wk * g["pred"].values).sum() / wk.sum()) if wk.sum() > 0 else float(g["pred"].mean())
        scheme = "shock" if (shock and shock_w_available) else "ordinary"
    WEIGHT_CACHE[(q_end, v)] = w
    out_rows.append({"date": q_end, "Vintage": v, "is_shock": shock, "breadth": int(g.merge(regime, on=["date","Vintage"])["breadth"].iloc[0]),
                     "weights": scheme, "actual": g["actuals"].iloc[0], "regime_aware_nowcast": pred})
ra = pd.DataFrame(out_rows); ra["error"] = ra["regime_aware_nowcast"] - ra["actual"]
with pd.ExcelWriter(f"{PATH_OUTPUT}/{date_str} 4_rslt regime_aware_nowcast.xlsx") as xw:
    ra.to_excel(xw, sheet_name="nowcast", index=False)
    regime.to_excel(xw, sheet_name="regime_log", index=False)
print(f"[OK] regime-aware nowcast written; weight schemes used: {ra['weights'].value_counts().to_dict()}")
print(ra[ra["is_shock"]][["date", "Vintage", "breadth", "weights", "regime_aware_nowcast", "error"]].round(2).to_string(index=False) if ra["is_shock"].any() else "[regime] no shock-regime quarter in the test window")

# =========================================================
# Beta14a dispersion bands — regime-conditional inverse-RMSE weighted
# quantiles of the (spec, estimator, seed) prediction pool.
# Replaces the split-conformal intervals from beta14.
# =========================================================
import pandas as pd, numpy as np

fcst_d = pd.read_excel(f"{PATH_OUTPUT}/{forecast_file}", parse_dates=["date"])
fcst_d = reading_guard(fcst_d[pool_mask(fcst_d)], label="bands")   # pool filter (cell 2) + reading guard (2026-10-25)
ra_d   = pd.read_excel(f"{PATH_OUTPUT}/{date_str} 4_rslt regime_aware_nowcast.xlsx", sheet_name="nowcast", parse_dates=["date"])

# beta15 (2026-10-18): the bands use the regime and the expanding-window weights of cell 23 (WEIGHT_CACHE),
# so consensus and bands are always computed under the same regime and the same information set.
VINTAGES_D = dict(VINTAGES)      # cell 2 (any grid, 2026-10-25)
id_vars = ["date", "spec", "estimator", "actuals"] + (["seed"] if "seed" in fcst_d.columns else [])
long_d = fcst_d.melt(id_vars=id_vars, value_vars=list(VINTAGES_D), var_name="horizon", value_name="pred")
long_d["Vintage"] = long_d["horizon"].map(VINTAGES_D)
long_d = long_d.dropna(subset=["pred"])

def _weighted_quantile(values, weights, q):
    """Weighted percentile, q in [0, 1]. Sort by value, cum-sum weights, interpolate."""
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & (weights > 0)
    if mask.sum() == 0:
        return np.nan
    v, w = values[mask], weights[mask]
    order = np.argsort(v)
    v, w = v[order], w[order]
    cw = np.cumsum(w)
    cw /= cw[-1]
    return float(np.interp(q, cw, v))

Q_LEVELS = {"p025": 0.025, "p10": 0.10, "p25": 0.25, "p50": 0.50,
            "p75": 0.75, "p90": 0.90, "p975": 0.975}

rows_d = []
for (date, V), g in long_d.groupby(["date", "Vintage"]):
    w = WEIGHT_CACHE.get((pd.Timestamp(date), V))
    weights = np.ones(len(g)) if w is None else np.array([w.get((s, e), 0.0) for s, e in zip(g["spec"], g["estimator"])])
    preds = g["pred"].values.astype(float)
    row = {"date": date, "Vintage": V, "is_shock": bool(ra.loc[(ra["date"] == pd.Timestamp(date)) & (ra["Vintage"] == V), "is_shock"].iloc[0]),
           "actual": g["actuals"].iloc[0]}
    for lbl, q in Q_LEVELS.items():
        row[lbl] = _weighted_quantile(preds, weights, q)
    rows_d.append(row)

bands = pd.DataFrame(rows_d).merge(
    ra_d[["date", "Vintage", "regime_aware_nowcast"]], on=["date", "Vintage"], how="left"
)

# Backward-compat aliases so existing chart cells keep working with a filename swap only
bands["lo_80"] = bands["p10"]
bands["hi_80"] = bands["p90"]
bands["lo_95"] = bands["p025"]
bands["hi_95"] = bands["p975"]

out_cols = ["date", "Vintage", "actual", "regime_aware_nowcast", "is_shock",
            "p025", "p10", "p25", "p50", "p75", "p90", "p975",
            "lo_80", "hi_80", "lo_95", "hi_95"]
out_path_d = f"{PATH_OUTPUT}/{date_str} 4_rslt dispersion_bands.xlsx"
bands[out_cols].to_excel(out_path_d, index=False)
print(f"[OK] Wrote {len(bands)} rows to {out_path_d}")

_shock_q = bands[bands["is_shock"]]["date"].unique()
for _d in _shock_q:
    print(f"\n{pd.Timestamp(_d):%Y-%m} (shock regime) dispersion bands (weighted percentiles):")
    print(bands[bands["date"] == _d][["Vintage", "regime_aware_nowcast", "p025", "p10", "p50", "p90", "p975"]].round(2).to_string(index=False))


## 9. Live quarter


In [ ]:
# [tto:live]
# =========================================================
# beta15 live quarter (2026-10-19): the out-of-sample reading goes through the same regime, cluster and band
# machinery as the backtest. One reading date: the data cut (last month with monthly data), which fixes the vintage.
# Uses breadth_at / _thr / MIN_BROKEN_FOR_ALERT / weights_for / PANDEMIC_RECORD, WATCHLIST / _family and
# _weighted_quantile / Q_LEVELS defined above in this cell. The LSTM has no out-of-sample reading, so the live
# pool is the sklearn cells only and the weights are renormalised over them.
# =========================================================
if desired_date is None:
    print('[live] no live quarter in this run (identification run)')
else:
    import pandas as pd, numpy as np

    oos = pd.read_excel(f"{PATH_OUTPUT}/{outofsample_file}", parse_dates=["date"])
    oos = reading_guard(oos[pool_mask(oos)].rename(columns={"nowcast": "pred"}), cols=["pred"], label="live").dropna(subset=["pred"])
    _monthly_cols = [c for c in data.columns if c != target_variable]
    data_cut = data.index[data[_monthly_cols].notna().any(axis=1)].max()           # last month with any monthly data
    # 2026-10-25: every unprinted quarter in the out-of-sample table is read at its own vintage (Q2 at V+4 and Q3 at V+1
    # with data to October); a vintage beyond the grid is capped at its last vintage. One row per quarter in the live file.
    _oos_all = oos.copy(); _live_rows, _live_cluster_tabs = [], []
    _pseudo_dates = [pd.Timestamp(k) for k in cfg.PSEUDO_PRINTS]
    for live_date in sorted(pd.to_datetime(_oos_all["date"].unique())):
        if live_date <= pd.Timestamp(test_end_date) or live_date in _pseudo_dates:
            continue                                                               # printed (or pseudo-printed) quarters are not live
        oos = _oos_all[_oos_all["date"] == live_date].copy()
        live_V = (data_cut.year - live_date.year) * 12 + (data_cut.month - live_date.month)
        _V_eff = min(live_V, max(lags))                                          # weights, record and clusters of the nearest backtested vintage
        live_q = f"{live_date.year}-Q{(live_date.month - 1) // 3 + 1}"
        print(f"[live] {live_q} read at V{live_V:+d} (data to {data_cut:%Y-%m}; record of V{_V_eff:+d}); {oos[['spec', 'estimator']].drop_duplicates().shape[0]} cells, {len(oos)} readings")

        # ---- regime at the live reading: breadth, clusters, trigger ----
        live_breadth = breadth_at(live_date, _V_eff)
        oos["method"], oos["K"], oos["variant"] = zip(*oos["spec"].apply(_family))
        live_pooled = float(oos["pred"].median())
        _rows = []
        for name, (V, meths, vars_, ests, (klo, khi), note, can_alert) in WATCHLIST.items():
            m = oos["method"].isin(meths) & oos["variant"].isin(vars_)
            if ests is not None: m &= oos["estimator"].isin(ests)
            if klo is not None:  m &= oos["K"].between(klo, khi)
            g = oos[m]
            if g.empty:
                print(f"[live] {name}: no cells in the live pool"); continue
            med = float(g["pred"].median())
            _rows.append({"date": live_date, "Vintage": live_V, "cluster": name, "n": int(len(g)), "median": med, "pooled_median": live_pooled,
                          "divergence": med - live_pooled,
                          "flag": (-1 if med - live_pooled <= -cfg.ALERT_PP else (1 if med - live_pooled >= cfg.ALERT_PP else 0)) * int(can_alert),
                          "alert": int(((med - live_pooled <= -cfg.ALERT_PP) if cfg.ALERT_SIDE == "below" else (abs(med - live_pooled) >= cfg.ALERT_PP)) and can_alert), "read_at": f"V{live_V:+d}", "defined_at": ("all" if V is None else f"V{V:+d}")})
        live_clusters = pd.DataFrame(_rows)
        live_alert = bool(live_clusters["alert"].any()) if len(live_clusters) else False
        trig_breadth = bool(cfg.BREADTH_TRIGGER) and (live_breadth >= _thr)
        trig_watch = live_alert and (live_breadth >= MIN_BROKEN_FOR_ALERT)
        live_shock = bool(trig_breadth or trig_watch)
        _k = live_date.strftime("%Y-%m-%d")
        if _k in REGIME_OVERRIDE["add"]:    live_shock = True
        if _k in REGIME_OVERRIDE["remove"]: live_shock = False
        live_trigger = "both" if (trig_breadth and trig_watch) else ("breadth" if trig_breadth else ("watchlist" if trig_watch else ""))

        # ---- weights at the live vintage (same function as the backtest), nowcast and bands ----
        w = weights_for(live_date, _V_eff, shock=True) if live_shock else None
        scheme = "shock" if w is not None else None
        if w is None:
            w = weights_for(live_date, _V_eff, shock=False); scheme = "ordinary" if w is not None else "equal"
        wk = np.ones(len(oos)) if w is None else np.array([w.get((s, e), 0.0) for s, e in zip(oos["spec"], oos["estimator"])])
        if wk.sum() == 0: wk = np.ones(len(oos)); scheme = "equal"
        live_nowcast = float((wk * oos["pred"].values).sum() / wk.sum())
        _q = {lbl: _weighted_quantile(oos["pred"].values.astype(float), wk, q) for lbl, q in Q_LEVELS.items()}
        live_prints = sum(pd.Timestamp(d) <= known_actual_cutoff(live_date, _V_eff) for d in test_q)
        live_row = pd.DataFrame([{"date": live_date, "quarter": live_q, "Vintage": live_V, "data_cut": data_cut, "is_shock": live_shock, "breadth": int(live_breadth),
                                  "threshold": _thr, "n_panel": len(_vars), "trigger": live_trigger, "watch_alert": live_alert, "weights": scheme,
                                  "prints_known": int(live_prints), "n_cells": int(oos[["spec", "estimator"]].drop_duplicates().shape[0]),
                                  "regime_aware_nowcast": live_nowcast, "pooled_median": live_pooled,
                                  "lo_80": _q["p10"], "hi_80": _q["p90"], "lo_95": _q["p025"], "hi_95": _q["p975"]}])
        _live_rows.append(live_row); _live_cluster_tabs.append(live_clusters)
        print(f"[live] breadth {live_breadth} of {len(_vars)} (threshold {_thr}); cluster alert {live_alert}; regime {'SHOCK' if live_shock else 'ordinary'}"
              f"{' (' + live_trigger + ')' if live_trigger else ''}; weights {scheme} on {live_prints} prints")
        print(f"[live] regime-aware nowcast {live_nowcast:+.2f}; pooled median {live_pooled:+.2f}; 80% band [{_q['p10']:+.2f}, {_q['p90']:+.2f}]; 95% band [{_q['p025']:+.2f}, {_q['p975']:+.2f}]")
        print(live_clusters[["cluster", "n", "median", "pooled_median", "divergence", "flag", "alert", "read_at", "defined_at"]].round(2).to_string(index=False))
    if _live_rows:
        with pd.ExcelWriter(f"{PATH_OUTPUT}/{date_str} 4_rslt live_quarter.xlsx") as xw:
            pd.concat(_live_rows, ignore_index=True).to_excel(xw, sheet_name="nowcast", index=False)
            pd.concat(_live_cluster_tabs, ignore_index=True).to_excel(xw, sheet_name="clusters", index=False)
        print(f"[live] {len(_live_rows)} live quarter(s) written")
    else:
        print("[live] no unprinted quarter in the out-of-sample table")
